# UC-07: Visualisasi I/O Produksi End-to-End + Suggestion Engine

**Sumber Data:** WIM/Ucan (weighbridge) + Urise (RFID/Geofence trip events) + Uscavis (stock mutation North Stockpile)

**Karakteristik:** Batch latency (refresh per jam) + Suggestion engine harian (Mosaic AI)

**Konsumen:** Direksi, Manajer Produksi, Tim Perencanaan, Manajer Crushing Plant, Manajer Quality, Tim Shipping

---

## Tujuan

Rekonsiliasi end-to-end produksi batu bara: produksi per ROM, per hauling (WIM/Ucan),
stock di North Stockpile (Uscavis), tonase yang masuk Crushing Plant, dan tonase yang
dikirim ke shipment. Ditambah **Suggestion Engine** berbasis Mosaic AI yang memberi
rekomendasi proaktif harian (balancing kontraktor, deteksi loss tidak wajar, prediksi
shortfall shipment).

## Funnel produksi (bercabang, bukan linear murni)

```
                                            +--> [4a] DIRECT ke CP ------------+
[1] ROM PRODUCTION --> [2] HAULING (WIM) ---+                                  +--> [5] CP INTAKE --> [6] SHIPMENT
                                            +--> [3a] STOCKPILE IN             |
                                                     |                         |
                                                     v                         |
                                                 [3b] STOCKPILE OUT (reclaim) -+
```

| # | Stage | Sumber otoritatif | Keterangan |
|---|-------|-------------------|------------|
| 1 | `ROM_PRODUCTION` | Urise (event ROM-A1/A2/A3, `alat_muat` EXCA/TLS) | tonase muat di ROM |
| 2 | `HAULING_WEIGHED` | WIM/Ucan (`netto`) | jembatan timbang KM-13 / closing gate = angka paling akurat |
| 3a | `STOCKPILE_IN` | Uscavis `movement = IN` | masuk North Stockpile |
| 3b | `STOCKPILE_OUT` | Uscavis `movement = OUT` | reclaim stockpile -> CP |
| 4a | `HAULING_DIRECT_CP` | WIM `last_port_name` = BIB CP x | ROM langsung ke CP tanpa singgah stockpile |
| 5 | `CP_INTAKE` | Urise (event RFID di CP-1..CP-9) | dumping di hopper CP |
| 6 | `SHIPMENT` | **UC-06** `uc.uscavis.gold_wim_cv_ratio.conveyor_tonnage` | tonase conveyor BLC (direct + non-direct) menuju shipment |

> **Sambungan ke UC-06.** Ketiga sumber UC-07 berhenti di CP intake, tapi sisi
> hilirnya sudah dihitung UC-06 dari Conveyor BLC. Stage 7 karena itu dibaca dari
> gold UC-06, bukan diestimasi. UC-06 sudah mengalikan volume conveyor dengan
> density BIB `0.88`, jadi angkanya sudah dalam ton. Bila tabel UC-06 belum ada,
> notebook menulis estimasi dengan flag `is_estimated = true` supaya dashboard
> tidak menampilkan estimasi sebagai angka aktual.
>
> `gold_cp_intake_daily` juga di-cross-check ke `uc.uscavis.gold_output_summary`
> (output totalizer CP per shift) untuk mendapat `crushing_recovery_pct` =
> output CP / intake CP.

## Arsitektur medallion

```
uc.wim.closing_transaction_25_july_2_agustus --> bronze_wim_ucan        --+
Urise transaksi   (tabel UC / transaksi.csv) --> bronze_urise_transaksi --+--> SILVER --> GOLD --> Dashboard
Uscavis stock  (tabel UC / stock_transaction) -> bronze_uscavis_stock   --+                 |
                                                                                            +--> Suggestion Engine (Mosaic AI)
uc.uscavis.gold_wim_cv_ratio (UC-06) ----------------------------------> stage SHIPMENT     +--> Arsip PostgreSQL (> 3 bulan)
uc.uscavis.gold_output_summary (UC-06) --------------------------------> cross-check CP
```

Output ditulis ke **`uc.uc07`** (satu schema, layer dibedakan prefix nama tabel),
mengikuti konvensi UC-06 yang memakai `uc.uscavis`. Label shift (`Pagi`/`Malam`)
dan kolom `adjusted_date` dibuat identik dengan UC-06 supaya gold kedua use case
bisa di-join langsung di dashboard.

**Silver:** `silver_wim_hauling`, `silver_urise_events`, `silver_uscavis_stock_txn`,
`silver_trip_reconciliation`

**Gold (dipakai dashboard):** `gold_funnel_stage_daily`, `gold_stage_reconciliation`,
`gold_production_timeseries`, `gold_contractor_daily`, `gold_rom_daily`,
`gold_cp_intake_daily`, `gold_stockpile_balance_hourly`, `gold_loss_anomaly`,
`gold_shipment_outlook`, `gold_suggestion_daily`, `gold_kpi_snapshot`

---
## 1. Konfigurasi

In [0]:
# ============================================================
# CONFIGURATION - UC07 End-to-End Production I/O
# ============================================================

# ---- Unity Catalog target ----
# Mengikuti konvensi UC-06 (uc.uscavis): satu schema per use case,
# layer dibedakan lewat prefix nama tabel (bronze_ / silver_ / gold_).
CATALOG = "uc"
UC07_SCHEMA = "uc07"
SCHEMAS = {"bronze": UC07_SCHEMA, "silver": UC07_SCHEMA, "gold": UC07_SCHEMA}

# ---- Source: tabel Unity Catalog (prioritas) ----
# WIM/Ucan sudah terdaftar di UC dan dipakai juga oleh UC-06.
SRC_WIM_TABLE     = "uc.wim.closing_transaction_25_july_2_agustus"
# Urise & Uscavis: isi bila sudah ada di UC. Biarkan None -> jalankan sel
# discovery di bagian 2.0, atau notebook otomatis fallback ke file.
SRC_URISE_TABLE   = "uc.urise.urise_transaction_june"  # Urise trip events (Juni; May-Jun period)
# NOTE: Urise data covers May 31 - Jun 20, WIM covers Jul 22 - Aug 2.
# Urise-dependent stages (ROM_PRODUCTION, CP_INTAKE) will be sparse for the WIM period.
# When July Urise data becomes available, update this reference accordingly.
SRC_USCAVIS_TABLE = "uc.urise.stock_transaction"       # Uscavis stock mutation (di schema uc.urise)

# ---- Source: file (fallback bila tabel UC belum ada) ----
SOURCE_ROOT = "/Volumes/uc/urise/urise_etc"
WIM_XLSX_PATH     = f"{SOURCE_ROOT}/Data Ucan 25 July - 2 Agustus.xlsx"   # WIM / Ucan (fallback only)
URISE_CSV_PATH    = f"{SOURCE_ROOT}/transaksi.csv"                        # fallback if UC table unavailable
USCAVIS_CSV_PATH  = f"{SOURCE_ROOT}/stock_transaction.csv"                # fallback if UC table unavailable

# Bronze hanya ditulis ulang bila sumbernya file. Bila sumbernya tabel UC,
# tabel sumber itu sendiri yang berperan sebagai Bronze (pola yang dipakai UC-06).
WRITE_BRONZE_COPY_FOR_FILES = True

# ---- Time & shift ----
TIMEZONE = "Asia/Makassar"          # WITA (UTC+8); WIM menyimpan waktu lokal WITA
SHIFT_START_HOUR = 6                # hari produksi mulai 06:00 lokal
SHIFT_LENGTH_HOURS = 12             # Pagi 06:00-17:59, Malam 18:00-05:59
# Label shift dibuat identik dengan UC-06 ("Pagi"/"Malam") supaya tabel gold
# kedua use case bisa di-join langsung di dashboard.
SHIFT_LABEL_DAY = "Pagi"
SHIFT_LABEL_NIGHT = "Malam"

# ---- Unit conversion ----
KG_PER_TON = 1000.0                 # tonase/netto/quantity di ketiga sumber = KILOGRAM
DENSITY_REFERENCE = 0.88            # density BIB dipakai UC-06 untuk volume conveyor -> ton;
                                    # angka conveyor yang dibaca UC-07 SUDAH dalam ton

# ---- Business thresholds ----
LOSS_PCT_WARN        = 2.0          # selisih tahap > 2%  -> perhatian
LOSS_PCT_CRITICAL    = 5.0          # selisih tahap > 5%  -> kritikal
PAYLOAD_MIN_TON      = 25.0         # netto < 25 t  -> anomali (under-load)
PAYLOAD_MAX_TON      = 80.0         # netto > 80 t  -> anomali (over-load / double weigh)
TRIP_MATCH_WINDOW_H  = 8            # window pencocokan WIM <-> event Urise (jam)
STOCKPILE_MIN_COVER_DAYS = 1.5      # buffer minimum North Stockpile (hari konsumsi CP)
OPENING_STOCK_TON    = 0.0          # saldo awal North Stockpile sebelum baris pertama Uscavis
                                    # (0 = saldo relatif; isi hasil survey stock opname bila ada)
CONTRACTOR_IMBALANCE_PCT = 15.0     # deviasi vs rata-rata fleet -> kandidat balancing
ANOMALY_Z_THRESHOLD  = 3.0          # robust z-score (MAD) untuk deteksi loss tak wajar

# ---- Production target (ton) ----
# Sumber sebenarnya: tabel RKAP / plan bulanan. Sementara dikonfigurasi di sini.
MONTHLY_TARGET_TON = 1_500_000.0
TARGET_BY_MONTH = {                 # override per bulan (format "YYYY-MM")
    "2026-07": 1_500_000.0,
    "2026-08": 1_550_000.0,
}
# Alokasi target per ROM (proporsi, total = 1.0)
ROM_TARGET_SHARE = {
    "ROM-A1": 0.28, "ROM-A2": 0.25, "ROM-A3": 0.20,
    "ROM-B1": 0.17, "ROM-B2": 0.06, "PIT-C-APL": 0.04,
}

# ---- Shipment (stage 7) - dari gold UC-06 ----
# gold_wim_cv_ratio.conveyor_tonnage = total tonase conveyor BLC per hari
# (direct CP->shipment + non-direct reclaim->shipment). Ini angka aktual yang
# paling dekat dengan tonase terkirim, bukan estimasi.
SHIPMENT_SOURCES = [
    {"table": "uc.uscavis.gold_wim_cv_ratio",
     "date_col": "date", "ton_col": "conveyor_tonnage",
     "label": "UC06 conveyor BLC (direct + non-direct)"},
    # cadangan: output totalizer CP per shift (lebih dekat ke CP output, bukan shipment)
    {"table": "uc.uscavis.gold_output_summary",
     "date_col": "date", "ton_col": "total_tonnage_ton",
     "label": "UC06 CP totalizer output"},
]
# Bila semua sumber di atas tak tersedia: tulis estimasi dari CP intake x recovery, di-flag.
SHIPMENT_FALLBACK_ENABLED = True
CRUSHING_RECOVERY_FACTOR  = 0.985    # asumsi recovery crushing (1.5% moisture/spillage)

# ---- Cross-check ke gold UC-06 (opsional, di-skip bila tabel tak ada) ----
UC06_CP_OUTPUT_TABLE = "uc.uscavis.gold_output_summary"   # date, shift, cp_name, total_tonnage_ton
UC06_WIM_CV_RATIO_TABLE = "uc.uscavis.gold_wim_cv_ratio"  # date, wim_tonnage, conveyor_tonnage, ratio

# ---- Mosaic AI suggestion engine ----
MOSAIC_AI_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
SUGGESTION_TOP_N   = 3
SUGGESTION_ENABLED = True            # False -> hanya rule engine (tanpa panggil LLM)

# ---- Retention / arsip ----
DELTA_RETENTION_MONTHS = 3           # window panas di Delta
PG_ARCHIVE_ENABLED = False           # True untuk mengaktifkan arsip PostgreSQL
PG_JDBC_URL   = "jdbc:postgresql://<host>:5432/<db>"
PG_SCHEMA     = "uc07_archive"
PG_SECRET_SCOPE = "bib-uc07"
PG_USER_KEY   = "pg-user"
PG_PASS_KEY   = "pg-password"

# ---- Incremental batch (refresh per jam) ----
# None = full reload (initial load). Isi timestamp untuk incremental.
INCREMENTAL_WATERMARK = None
FULL_REFRESH = True

print("=" * 70)
print("UC07 - VISUALISASI I/O PRODUKSI END-TO-END + SUGGESTION ENGINE")
print("=" * 70)
print(f"  Output           : {CATALOG}.{UC07_SCHEMA}")
print(f"  Sumber WIM       : {SRC_WIM_TABLE}")
print(f"  Sumber Urise     : {SRC_URISE_TABLE}")
print(f"  Sumber Uscavis   : {SRC_USCAVIS_TABLE}")
print(f"  Shipment         : {SHIPMENT_SOURCES[0]['table']}.{SHIPMENT_SOURCES[0]['ton_col']}")
print(f"  Timezone         : {TIMEZONE} (hari produksi mulai {SHIFT_START_HOUR:02d}:00)")
print(f"  Target bulanan   : {MONTHLY_TARGET_TON:,.0f} ton")
print(f"  Mosaic endpoint  : {MOSAIC_AI_ENDPOINT} (enabled={SUGGESTION_ENABLED})")
print(f"  Retensi Delta    : {DELTA_RETENTION_MONTHS} bulan | arsip PG: {PG_ARCHIVE_ENABLED}")

### 1.1 Referensi master data

Mapping kode kontraktor, normalisasi nama ROM, dan normalisasi nama port/tujuan.
Diturunkan dari profiling ketiga sumber (WIM `code`/`hauling_contractor`,
Urise `kontraktor`, prefix `no_lambung`/`truck`).

In [0]:
# Kode kontraktor (prefix nomor lambung / kolom `code` di WIM) -> nama badan usaha
CONTRACTOR_MAP = {
    "GEC":  "PT Goden Energi Cemerlang Lestari",
    "KMB":  "KSU Mitra Bersatu",
    "BMT":  "PT Bumiputera Maha Terpecaya",
    "RBT":  "PT Rezki Batulicin Transport",
    "BBS":  "PT Borneo Banua Sejahtera",
    "MMS":  "PT Makmur Mulya Sebamban",
    "MAV":  "PT. Multi Adverindo",
    "AEK":  "PT Anugrah Energi Kalimantan",
    "BKA":  "PT Barajasa Kalimantan Abadi Energi",
    "RAM":  "PT Rizky Aulia Mandiri",
    "RAMB": "PT Rizky Aulia Mandiri",
    "TMR":  "PT. Tama Mulia Resources",       # fleet reclaim stockpile -> CP
    "EST":  "PT Energi Sinar Tambang",
    "PDT":  "PT Pandu Dwi Tunggal",
}

# Fleet yang beroperasi di dalam area stockpile (bukan hauling ROM -> port)
STOCKPILE_FLEET_PREFIXES = ["TMR", "PDT", "EST"]

# Daftar Crushing Plant aktif
CP_LIST = ["CP-1", "CP-2", "CP-2A", "CP-2B", "CP-3", "CP-4",
           "CP-5", "CP-6", "CP-7", "CP-8", "CP-9"]

# Daftar ROM (setelah normalisasi)
ROM_LIST = ["ROM-A1", "ROM-A2", "ROM-A3", "ROM-B1", "ROM-B2", "PIT-C-APL"]

# Definisi stage funnel: (urutan, kode, label, cabang)
FUNNEL_STAGES = [
    (1,  "ROM_PRODUCTION",    "ROM Production",        "MAIN"),
    (2,  "HAULING_WEIGHED",   "Hauling (WIM/Ucan)",    "MAIN"),
    (3,  "HAULING_DIRECT_CP", "Direct ROM -> CP",      "DIRECT"),
    (4,  "STOCKPILE_IN",      "Stockpile IN",          "BUFFER"),
    (5,  "STOCKPILE_OUT",     "Stockpile OUT/Reclaim", "BUFFER"),
    (6,  "CP_INTAKE",         "Crushing Plant Intake", "MAIN"),
    (7,  "SHIPMENT",          "Shipment",              "MAIN"),
]

# Kolom WIM/Ucan yang dipakai (dari 124 kolom sumber, sisanya di-drop di Bronze)
WIM_COLUMNS = [
    "id", "truck", "driver", "client", "material", "hauling", "coal", "code",
    "timestamp_gross_local", "timestamp_gross_utc",
    "timestamp_in_local", "timestamp_in_utc",
    "timestamp_out_local", "timestamp_out_utc",
    "gross", "netto", "tare", "netto_ucan", "netto_ori",
    "data_source", "location_in", "location_closing",
    "last_rom_name", "last_rom_date_time", "last_hauling_date_time",
    "last_port_name", "last_port_date_time", "last_km13_date_time",
    "hauling_contractor", "truck_type", "total_gandar", "average_speed",
    "route_distance", "status_trx_wim", "doc", "flag_line",
    "adjustment_result", "transaction_validation",
    "anomaly_message_in", "anomaly_message_create", "anomaly_message_out",
    "rfid", "rfid_wim", "created_date",
]

print(f"Kontraktor terdaftar : {len(CONTRACTOR_MAP)}")
print(f"Crushing Plant       : {len(CP_LIST)}")
print(f"ROM                  : {len(ROM_LIST)}")
print(f"Stage funnel         : {len(FUNNEL_STAGES)}")

### 1.2 Import & helper

In [0]:
import json
import math
import re
import warnings
from datetime import datetime, date, timedelta
from typing import Dict, List, Optional, Tuple

import pandas as pd

import pyspark
import pyspark.sql.functions as F
from pyspark.sql import DataFrame, Window
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType,
    IntegerType, BooleanType, TimestampType, DateType,
)
from pyspark.sql.utils import AnalysisException

warnings.filterwarnings("ignore")

spark.conf.set("spark.sql.session.timeZone", "UTC")   # semua timestamp internal = UTC
# NOTE: schema.autoMerge not supported on Serverless; handled via .option("mergeSchema", "true") in write_table
# NOTE: AQE is always enabled on Serverless; no need to set explicitly

RUN_TS = datetime.utcnow()
print(f"PySpark {pyspark.__version__} | run_ts_utc = {RUN_TS:%Y-%m-%d %H:%M:%S}")

In [0]:
# ============================================================
# HELPERS
# ============================================================

def tbl(layer: str, name: str) -> str:
    """Fully-qualified Unity Catalog table name."""
    return f"{CATALOG}.{SCHEMAS[layer]}.{name}"


def write_table(df: DataFrame, layer: str, name: str,
                mode: str = "overwrite",
                partition_by: Optional[List[str]] = None,
                comment: Optional[str] = None,
                zorder_by: Optional[List[str]] = None) -> str:
    """Persist a DataFrame as a managed Delta table and return its FQN."""
    fqn = tbl(layer, name)
    writer = (df.write.format("delta").mode(mode)
              .option("overwriteSchema", "true")
              .option("mergeSchema", "true"))
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.saveAsTable(fqn)

    if comment:
        spark.sql(f"COMMENT ON TABLE {fqn} IS '{comment.replace(chr(39), chr(39) * 2)}'")
    if zorder_by:
        spark.sql(f"OPTIMIZE {fqn} ZORDER BY ({', '.join(zorder_by)})")

    n = spark.table(fqn).count()
    print(f"  [{layer.upper():6}] {fqn:<55} {n:>10,} rows")
    return fqn


def merge_upsert(df: DataFrame, layer: str, name: str, keys: List[str]) -> str:
    """Idempotent upsert - dipakai untuk refresh per jam (incremental)."""
    fqn = tbl(layer, name)
    if not spark.catalog.tableExists(fqn):
        return write_table(df, layer, name)
    df.createOrReplaceTempView("_stg_upsert")
    on_clause = " AND ".join([f"t.{k} <=> s.{k}" for k in keys])
    spark.sql(f"""
        MERGE INTO {fqn} t
        USING _stg_upsert s ON {on_clause}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"  [{layer.upper():6}] MERGE -> {fqn} on {keys}")
    return fqn


def to_ton(col):
    """Kilogram -> ton, dibulatkan 3 desimal."""
    return F.round(F.col(col).cast("double") / F.lit(KG_PER_TON), 3)


def clean_null(col):
    """Sumber memakai '\\N', 'null', '' sebagai penanda kosong."""
    c = F.trim(F.col(col).cast("string"))
    return F.when(c.isin("\\N", "\\\\N", "null", "NULL", "NaN", "", "None"), F.lit(None)).otherwise(c)


def add_time_columns(df: DataFrame, ts_utc_col: str, prefix: str = "") -> DataFrame:
    """Tambah kolom waktu lokal, tanggal produksi (mulai 06:00), shift, jam, minggu, bulan."""
    p = prefix
    local = f"from_utc_timestamp({ts_utc_col}, '{TIMEZONE}')"
    return (
        df
        .withColumn(f"{p}ts_local", F.expr(local))
        .withColumn(f"{p}date_local", F.expr(f"to_date({local})"))
        .withColumn(f"{p}hour_local", F.expr(f"hour({local})"))
        .withColumn(f"{p}production_date",
                    F.expr(f"to_date({local} - INTERVAL {SHIFT_START_HOUR} HOURS)"))
        .withColumn(f"{p}shift",
                    F.expr(f"CASE WHEN hour({local}) >= {SHIFT_START_HOUR}"
                           f" AND hour({local}) < {SHIFT_START_HOUR + SHIFT_LENGTH_HOURS}"
                           f" THEN '{SHIFT_LABEL_DAY}' ELSE '{SHIFT_LABEL_NIGHT}' END"))
        # alias nama kolom UC-06 supaya gold UC-06 dan UC-07 bisa di-join langsung
        .withColumn(f"{p}adjusted_date", F.col(f"{p}production_date"))
        .withColumn(f"{p}week_start",
                    F.expr(f"date_trunc('WEEK', {local})").cast("date"))
        .withColumn(f"{p}month", F.expr(f"date_format({local}, 'yyyy-MM')"))
    )


def norm_rom(col):
    """'ROM A1 EXTEND' / 'ROM A2P' / 'ROM A3 GH' / 'PIT C APL' -> kode ROM standar."""
    c = F.upper(F.trim(col.cast("string")))
    return (
        F.when(c.rlike(r"^ROM\s*A\s*1"), F.lit("ROM-A1"))
         .when(c.rlike(r"^ROM\s*A\s*2"), F.lit("ROM-A2"))
         .when(c.rlike(r"^ROM\s*A\s*3"), F.lit("ROM-A3"))
         .when(c.rlike(r"^ROM\s*B\s*1"), F.lit("ROM-B1"))
         .when(c.rlike(r"^ROM\s*B\s*2"), F.lit("ROM-B2"))
         .when(c.rlike(r"PIT\s*C"),      F.lit("PIT-C-APL"))
         .when(c.rlike(r"^ROM-A1$|^ROM-A2$|^ROM-A3$"), c)
         .otherwise(F.lit("UNKNOWN"))
    )


def norm_port(col):
    """'BIB CP 7' / 'CP-2B' / 'NORTH STOCKPILE' / 'PORT BIB' -> kode tujuan standar."""
    c = F.upper(F.trim(col.cast("string")))
    cp_num = F.regexp_extract(c, r"CP[\s\-_]*([0-9]+[AB]?)", 1)
    return (
        F.when(c.rlike(r"NORTH[\s\-]*STOCK"), F.lit("NORTH-STOCKPILE"))
         .when(c.rlike(r"STOCKPILE"),          F.lit("NORTH-STOCKPILE"))
         .when(cp_num != "", F.concat(F.lit("CP-"), cp_num))
         .when(c.rlike(r"PORT"), F.lit("PORT-BIB"))
         .otherwise(F.lit("UNKNOWN"))
    )


def contractor_from_truck(col):
    """Prefix nomor lambung ('GEC 9123') -> kode kontraktor."""
    return F.upper(F.regexp_extract(F.trim(col.cast("string")), r"^([A-Za-z]+)", 1))


CONTRACTOR_MAP_EXPR = F.create_map([F.lit(x) for kv in CONTRACTOR_MAP.items() for x in kv])


def contractor_name(code_col):
    return F.coalesce(CONTRACTOR_MAP_EXPR[code_col], F.lit("UNKNOWN"))


def cp_name_uc06(col):
    """'CP-2A' (konvensi UC-07) -> 'CP2A' (konvensi UC-06) untuk join lintas use case."""
    return F.regexp_replace(F.upper(F.trim(col.cast("string"))), r"^CP[\s\-_]*", "CP")


def safe_pct(numer, denom):
    """Persentase aman-pembagi-nol, 2 desimal."""
    return F.round(F.when(F.col(denom) > 0,
                          F.col(numer) / F.col(denom) * F.lit(100.0))
                    .otherwise(F.lit(None)), 2)


print("Helper siap: tbl, write_table, merge_upsert, to_ton, clean_null, "
      "add_time_columns, norm_rom, norm_port, contractor_name, cp_name_uc06, safe_pct")

In [0]:
# Schema output (catalog `uc` diasumsikan sudah ada dan dikelola admin platform)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{UC07_SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
print(f"  schema output siap: {CATALOG}.{UC07_SCHEMA}")

---
## 2. BRONZE LAYER - Ingestion & normalisasi tipe

Sumber diambil dari **tabel Unity Catalog bila sudah ada** (pola yang sama dengan
UC-06 yang membaca `uc.uscavis_raw.*`, `uc.synova_raw.*`, `uc.wim.*`), dan jatuh ke
file CSV/Excel hanya kalau tabelnya belum terdaftar.

Bronze tetap dimaterialisasi ke `uc.uc07.bronze_*` walau sumbernya sudah tabel UC,
karena di sini dilakukan normalisasi yang tidak ada di tabel mentah:

- `id` WIM dibaca sebagai **string** (snowflake 18 digit; kalau lewat float,
  28.065 baris menyusut jadi 264 id unik),
- penanda kosong `\N` (WIM) dan literal `'null'` (CSV Urise/Uscavis) -> NULL sejati,
- timestamp ISO-8601 `...Z` di-parse jadi TIMESTAMP UTC,
- `timestamp_gross_utc` diturunkan dari waktu lokal bila kolom UTC tidak ada,
- metadata `_ingested_at`, `_source_origin`, `_source_system`.

### 2.0 Discovery tabel sumber di Unity Catalog

Nama tabel Urise dan Uscavis belum dipastikan. Sel ini menyisir catalog `uc`
dan mengisi `SRC_URISE_TABLE` / `SRC_USCAVIS_TABLE` otomatis bila ketemu.
Kalau tidak ketemu, isi manual di sel konfigurasi atau biarkan fallback ke file.

In [0]:
DISCOVERY_KEYWORDS = {
    "URISE":   ["transaksi", "urise", "trip", "geofence"],
    "USCAVIS": ["stock_transaction", "stock_txn", "stock_mutation", "stockpile"],
    "WIM":     ["closing_transaction", "wim", "ucan"],
}


def discover_tables(catalog: str = CATALOG) -> pd.DataFrame:
    """Daftar semua tabel di catalog beserta kecocokan kata kunci sumber UC-07."""
    rows = []
    try:
        schemas = [r[0] for r in spark.sql(f"SHOW SCHEMAS IN {catalog}").collect()]
    except Exception as e:                                  # noqa: BLE001
        print(f"[WARN] tidak bisa membaca schema di catalog {catalog}: "
              f"{type(e).__name__}: {str(e)[:120]}")
        return pd.DataFrame(columns=["table", "matches"])

    for sch in schemas:
        if sch in ("information_schema",):
            continue
        try:
            tables = spark.sql(f"SHOW TABLES IN {catalog}.{sch}").collect()
        except Exception:                                   # noqa: BLE001
            continue
        for t in tables:
            fq = f"{catalog}.{sch}.{t['tableName']}"
            low = fq.lower()
            hits = [src for src, kws in DISCOVERY_KEYWORDS.items()
                    if any(k in low for k in kws)]
            rows.append({"table": fq, "matches": ",".join(hits)})
    return pd.DataFrame(rows)


df_catalog = discover_tables()
print(f"Tabel terdeteksi di catalog {CATALOG}: {len(df_catalog)}")
if not df_catalog.empty:
    hits = df_catalog[df_catalog["matches"] != ""]
    print("\nKandidat sumber UC-07:")
    print(hits.to_string(index=False) if not hits.empty else "  (tidak ada yang cocok)")

    def _pick(src_key: str) -> Optional[str]:
        cand = df_catalog[df_catalog["matches"].str.contains(src_key, na=False)]
        return cand.iloc[0]["table"] if len(cand) == 1 else None

    if SRC_URISE_TABLE is None:
        SRC_URISE_TABLE = _pick("URISE")
        print(f"\nSRC_URISE_TABLE   -> {SRC_URISE_TABLE or '(tidak unik/ tidak ketemu, pakai file)'}")
    if SRC_USCAVIS_TABLE is None:
        SRC_USCAVIS_TABLE = _pick("USCAVIS")
        print(f"SRC_USCAVIS_TABLE -> {SRC_USCAVIS_TABLE or '(tidak unik/ tidak ketemu, pakai file)'}")

In [0]:
def table_exists(fqn: Optional[str]) -> bool:
    if not fqn:
        return False
    try:
        return spark.catalog.tableExists(fqn)
    except Exception:                                       # noqa: BLE001
        return False


def read_csv_source(path: str) -> DataFrame:
    """CSV mentah: semua kolom string, cast eksplisit menyusul."""
    return (spark.read
            .option("header", True)
            .option("inferSchema", False)
            .option("multiLine", True)
            .option("escape", '"')
            .csv(path))


def resolve_source(table_fqn: Optional[str], file_path: str,
                   reader, label: str) -> Tuple[DataFrame, str]:
    """Pakai tabel UC bila ada; kalau tidak, baca file. Kembalikan (df, asal)."""
    if table_exists(table_fqn):
        print(f"{label}: tabel UC  -> {table_fqn}")
        return spark.table(table_fqn), f"UC_TABLE:{table_fqn}"
    print(f"{label}: file       -> {file_path}"
          + (f"  (tabel {table_fqn} tidak ditemukan)" if table_fqn else ""))
    return reader(file_path), f"FILE:{file_path}"


def parse_ts(col_name: str):
    """Timestamp yang mungkin sudah TIMESTAMP, mungkin string ISO-8601 '...Z'."""
    c = F.col(col_name)
    as_string = F.regexp_replace(c.cast("string"), "Z$", "")
    return F.coalesce(
        c.cast("timestamp"),
        F.to_timestamp(as_string, "yyyy-MM-dd'T'HH:mm:ss[.SSS]"),
        F.to_timestamp(as_string, "yyyy-MM-dd HH:mm:ss[.SSS]"),
    )


print("Resolver siap: table_exists, read_csv_source, resolve_source, parse_ts")

### 2.1 WIM/Ucan -> `bronze_wim_ucan`

Satu baris = satu transaksi penimbangan truk (jembatan timbang KM-13 atau closing gate).
Kolom kunci: `netto` (kg), `last_rom_name` (asal), `last_port_name` (tujuan),
`timestamp_gross_utc` (waktu timbang), `truck` (nomor lambung).

Sumber utama `uc.wim.closing_transaction_25_july_2_agustus` - tabel yang sama
yang dipakai UC-06 untuk rasio WIM/conveyor, jadi angka hauling kedua use case
dijamin berasal dari satu sumber.

In [0]:
NUMERIC_WIM = ["gross", "netto", "tare", "netto_ucan", "netto_ori",
               "total_gandar", "average_speed", "route_distance"]
TIMESTAMP_WIM = [c for c in WIM_COLUMNS
                 if c.startswith("timestamp_") or c.endswith("_date_time")
                 or c == "created_date"]


def _local_path(p: str) -> str:
    """Path yang bisa dibaca pandas: dbfs:/x -> /dbfs/x, sisanya apa adanya."""
    return p.replace("dbfs:/", "/dbfs/") if p.startswith("dbfs:/") else p


def read_wim_excel(path: str) -> DataFrame:
    """Fallback: baca Excel via pandas. `id` WAJIB dibaca sebagai string."""
    pdf = pd.read_excel(
        _local_path(path),
        dtype={"id": str, "itws_id": str, "wim_id": str, "truck_id": str,
               "master_truck_id": str, "real_transaction_id": str},
        engine="openpyxl",
    )
    print(f"  Excel terbaca: {pdf.shape[0]:,} baris x {pdf.shape[1]} kolom")
    for c in [c for c in WIM_COLUMNS if c not in pdf.columns]:
        pdf[c] = None
    pdf = pdf[WIM_COLUMNS].copy()
    for c in NUMERIC_WIM:
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce")
    for c in TIMESTAMP_WIM:
        pdf[c] = pd.to_datetime(pdf[c], errors="coerce")
    for c in pdf.columns:
        if c not in NUMERIC_WIM and c not in TIMESTAMP_WIM:
            pdf[c] = pdf[c].astype("string")
    pdf = pdf.astype(object).where(pd.notnull(pdf), None)
    return spark.createDataFrame(pdf)


df_wim_src, wim_origin = resolve_source(
    SRC_WIM_TABLE, WIM_XLSX_PATH, read_wim_excel, "WIM/Ucan")

# Kolom yang tidak ada di sumber dibuat NULL supaya skema Bronze selalu sama
missing_wim = [c for c in WIM_COLUMNS if c not in df_wim_src.columns]
for c in missing_wim:
    df_wim_src = df_wim_src.withColumn(c, F.lit(None).cast("string"))
if missing_wim:
    print(f"  kolom tidak ada di sumber (diisi NULL): {missing_wim}")

df_wim_bronze = df_wim_src.select(*WIM_COLUMNS)

# Use try_cast for numeric (tolerates \N and other malformed values -> NULL)
for c in NUMERIC_WIM:
    df_wim_bronze = df_wim_bronze.withColumn(c, F.expr(f"try_cast(`{c}` as double)"))
for c in TIMESTAMP_WIM:
    df_wim_bronze = df_wim_bronze.withColumn(c, F.expr(f"try_cast(`{c}` as timestamp)"))
for c in df_wim_bronze.columns:
    if c not in NUMERIC_WIM and c not in TIMESTAMP_WIM:
        df_wim_bronze = df_wim_bronze.withColumn(c, clean_null(c))

# Sumber UC-06 hanya memakai timestamp_gross_local. Bila kolom UTC kosong,
# turunkan dari waktu lokal WITA supaya seluruh pipeline tetap berbasis UTC.
df_wim_bronze = (
    df_wim_bronze
    .withColumn("timestamp_gross_utc",
                F.coalesce(F.col("timestamp_gross_utc"),
                           F.expr(f"to_utc_timestamp(timestamp_gross_local, '{TIMEZONE}')")))
    .withColumn("timestamp_out_utc",
                F.coalesce(F.col("timestamp_out_utc"),
                           F.expr(f"to_utc_timestamp(timestamp_out_local, '{TIMEZONE}')")))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_origin", F.lit(wim_origin))
    .withColumn("_source_system", F.lit("WIM_UCAN"))
)

write_table(df_wim_bronze, "bronze", "bronze_wim_ucan",
            comment="Raw weighbridge WIM/Ucan (1 baris = 1 penimbangan truk); "
                    "sumber sama dengan UC-06 gold_wim_cv_ratio.")
display(df_wim_bronze.limit(5))

### 2.2 Urise -> `bronze_urise_transaksi`

Event stream Urise: RFID gate + geofence. Satu baris = satu kunjungan truk ke satu
`master_location` (ROM, NORTH-STOCKPILE, atau CP), dengan `room_in` / `room_out`.
Timestamp berformat ISO-8601 UTC (`...Z`), penanda kosong berupa literal `'null'`.

In [0]:
URISE_TS_COLS = ["room_in", "room_out", "created_at", "updated_at",
                 "validated_at", "deleted_at", "time_out_ucan"]

df_urise_bronze, urise_origin = resolve_source(
    SRC_URISE_TABLE, URISE_CSV_PATH, read_csv_source, "Urise")

for c in df_urise_bronze.columns:
    df_urise_bronze = df_urise_bronze.withColumn(c, clean_null(c))

for c in URISE_TS_COLS:
    if c in df_urise_bronze.columns:
        df_urise_bronze = df_urise_bronze.withColumn(c, parse_ts(c))

df_urise_bronze = (
    df_urise_bronze
    .withColumn("id", F.col("id").cast("long"))
    .withColumn("tonase", F.col("tonase").cast("double"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_origin", F.lit(urise_origin))
    .withColumn("_source_system", F.lit("URISE"))
)

write_table(df_urise_bronze, "bronze", "bronze_urise_transaksi",
            comment="Raw Urise trip events (RFID gate + geofence) di ROM, North Stockpile, dan CP.")
display(df_urise_bronze.limit(5))

### 2.3 Uscavis -> `bronze_uscavis_stock`

Mutasi stok North Stockpile dari Uscavis. `movement` = `IN` / `OUT`,
`quantity` dalam kg, `remarks` memuat nomor lambung truk
(`Auto stock mutation for TMR 0003`).

In [0]:
df_uscavis_bronze, uscavis_origin = resolve_source(
    SRC_USCAVIS_TABLE, USCAVIS_CSV_PATH, read_csv_source, "Uscavis")

for c in df_uscavis_bronze.columns:
    df_uscavis_bronze = df_uscavis_bronze.withColumn(c, clean_null(c))

df_uscavis_bronze = (
    df_uscavis_bronze
    .withColumn("id", F.col("id").cast("long"))
    .withColumn("quantity", F.col("quantity").cast("double"))
    .withColumn("transaction_date", parse_ts("transaction_date"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_origin", F.lit(uscavis_origin))
    .withColumn("_source_system", F.lit("USCAVIS"))
)

write_table(df_uscavis_bronze, "bronze", "bronze_uscavis_stock",
            comment="Raw Uscavis stock mutation North Stockpile (IN/OUT, quantity kg).")
display(df_uscavis_bronze.limit(5))

In [0]:
# Ringkasan bronze + asal sumber (lineage)
print(f"{'tabel':<28} {'baris':>10}  asal")
for name, origin in [("bronze_wim_ucan", wim_origin),
                     ("bronze_urise_transaksi", urise_origin),
                     ("bronze_uscavis_stock", uscavis_origin)]:
    n = spark.table(tbl("bronze", name)).count()
    print(f"{name:<28} {n:>10,}  {origin}")

---
## 3. SILVER LAYER - Cleansed & conformed

Aturan yang diterapkan di Silver:

1. **Deduplikasi** - WIM tidak punya primary key yang andal setelah pembacaan Excel,
   sehingga dibentuk `trip_id = sha2(truck || timestamp_gross_utc)`. Urise & Uscavis
   dedup dengan `id` sumber.
2. **Unit** - semua tonase dikonversi kg -> **ton**.
3. **Timezone** - semua timestamp disimpan UTC, kolom turunan `*_local`,
   `production_date` (hari produksi mulai 06:00 WITA), dan `shift`.
4. **Conformed dimension** - `rom_code`, `dest_code`, `contractor_code`,
   `contractor_name` memakai kamus yang sama di ketiga sumber sehingga bisa di-join.
5. **Quality flag** - payload di luar rentang wajar, anomaly message dari WIM,
   event tanpa tonase, dan event tanpa pasangan in/out ditandai (tidak dibuang),
   agar loss tidak "hilang diam-diam" dari rekonsiliasi.

### 3.1 `silver_wim_hauling` - transaksi timbang (sumber tonase otoritatif)

In [0]:
bronze_wim = spark.table(tbl("bronze", "bronze_wim_ucan"))

silver_wim = (
    bronze_wim
    # --- identitas trip ---
    .withColumn("trip_id", F.sha2(F.concat_ws("|",
                                              F.upper(F.trim(F.col("truck"))),
                                              F.col("timestamp_gross_utc").cast("string")), 256))
    .withColumn("wim_id_src", F.col("id"))
    .withColumn("truck", F.upper(F.trim(F.col("truck"))))
    # --- kontraktor ---
    .withColumn("contractor_code",
                F.coalesce(F.upper(F.trim(F.col("hauling_contractor"))),
                           F.upper(F.trim(F.col("code"))),
                           contractor_from_truck(F.col("truck"))))
    .withColumn("contractor_name", contractor_name(F.col("contractor_code")))
    # --- asal / tujuan ---
    .withColumn("rom_code", norm_rom(F.col("last_rom_name")))
    .withColumn("dest_code", norm_port(F.col("last_port_name")))
    .withColumn("route_code", F.col("hauling"))
    .withColumn("client", F.upper(F.trim(F.col("client"))))
    # --- material (format: BLOK.SEAM.KUALITAS.PRODUK, mis. RA3.GRMH1.BR.ST) ---
    .withColumn("material_code", F.col("material"))
    .withColumn("material_block", F.split(F.col("material"), r"\.").getItem(0))
    .withColumn("material_seam", F.split(F.col("material"), r"\.").getItem(1))
    .withColumn("material_quality", F.split(F.col("material"), r"\.").getItem(2))
    .withColumn("material_product", F.split(F.col("material"), r"\.").getItem(3))
    .withColumn("coal_type", F.col("coal"))          # RTP / PTP
    # --- tonase ---
    .withColumn("gross_ton", to_ton("gross"))
    .withColumn("tare_ton", to_ton("tare"))
    .withColumn("netto_ton", to_ton("netto"))
    .withColumn("netto_ucan_ton", to_ton("netto_ucan"))
    .withColumn("netto_delta_wim_ucan_ton",
                F.round(F.col("netto_ton") - F.col("netto_ucan_ton"), 3))
    # --- waktu (gross = waktu timbang, dipakai sebagai waktu trip) ---
    .withColumn("weigh_ts_utc", F.col("timestamp_gross_utc"))
    .withColumn("rom_exit_ts_utc",
                F.expr(f"to_utc_timestamp(last_rom_date_time, '{TIMEZONE}')"))
    .withColumn("hauling_start_ts_utc",
                F.expr(f"to_utc_timestamp(last_hauling_date_time, '{TIMEZONE}')"))
    .withColumn("port_arrival_ts_utc",
                F.expr(f"to_utc_timestamp(last_port_date_time, '{TIMEZONE}')"))
    .withColumn("gate_out_ts_utc", F.col("timestamp_out_utc"))
    # --- durasi siklus (menit) ---
    .withColumn("rom_to_weigh_min",
                F.round((F.col("weigh_ts_utc").cast("long")
                         - F.col("rom_exit_ts_utc").cast("long")) / 60.0, 1))
    .withColumn("weigh_to_port_min",
                F.round((F.col("port_arrival_ts_utc").cast("long")
                         - F.col("weigh_ts_utc").cast("long")) / 60.0, 1))
    .withColumn("rom_to_port_min",
                F.round((F.col("port_arrival_ts_utc").cast("long")
                         - F.col("rom_exit_ts_utc").cast("long")) / 60.0, 1))
    # --- jalur funnel ---
    .withColumn("flow_path",
                F.when(F.col("dest_code") == "NORTH-STOCKPILE", F.lit("VIA_STOCKPILE"))
                 .when(F.col("dest_code").startswith("CP-"), F.lit("DIRECT_CP"))
                 .when(F.col("dest_code") == "PORT-BIB", F.lit("DIRECT_PORT"))
                 .otherwise(F.lit("UNKNOWN")))
    # --- flag kualitas data ---
    .withColumn("is_payload_anomaly",
                (F.col("netto_ton") < F.lit(PAYLOAD_MIN_TON))
                | (F.col("netto_ton") > F.lit(PAYLOAD_MAX_TON)))
    .withColumn("has_wim_anomaly_msg",
                F.coalesce(F.col("anomaly_message_in"), F.lit("")) != "")
    .withColumn("is_manual_weigh", F.col("status_trx_wim") == F.lit("MANUAL"))
    .withColumn("is_adjusted", F.col("adjustment_result").isin("Adjust", "Reject"))
    .withColumn("is_rom_unknown", F.col("rom_code") == F.lit("UNKNOWN"))
    .withColumn("is_dest_unknown", F.col("dest_code") == F.lit("UNKNOWN"))
    .withColumn("_source_system", F.lit("WIM_UCAN"))
)

silver_wim = add_time_columns(silver_wim, "weigh_ts_utc")

silver_wim = (
    silver_wim
    .select(
        "trip_id", "wim_id_src", "truck", "truck_type", "driver",
        "contractor_code", "contractor_name", "client",
        "rom_code", "dest_code", "route_code", "flow_path",
        "material_code", "material_block", "material_seam",
        "material_quality", "material_product", "coal_type",
        "gross_ton", "tare_ton", "netto_ton", "netto_ucan_ton",
        "netto_delta_wim_ucan_ton",
        "weigh_ts_utc", "rom_exit_ts_utc", "hauling_start_ts_utc",
        "port_arrival_ts_utc", "gate_out_ts_utc",
        "rom_to_weigh_min", "weigh_to_port_min", "rom_to_port_min",
        "ts_local", "date_local", "hour_local", "production_date", "shift",
        "week_start", "month",
        "route_distance", "average_speed", "total_gandar",
        "data_source", "location_in", "location_closing", "doc",
        "status_trx_wim", "adjustment_result",
        "is_payload_anomaly", "has_wim_anomaly_msg", "is_manual_weigh",
        "is_adjusted", "is_rom_unknown", "is_dest_unknown",
        "_source_system", "_ingested_at",
    )
    # dedup: satu truk tidak bisa ditimbang dua kali pada detik yang sama
    .dropDuplicates(["trip_id"])
)

write_table(silver_wim, "silver", "silver_wim_hauling",
            partition_by=["production_date"],
            comment="Transaksi timbang WIM/Ucan yang sudah dibersihkan; 1 baris = 1 trip hauling.")

display(silver_wim.limit(10))

In [0]:
# Sanity check WIM
sw = spark.table(tbl("silver", "silver_wim_hauling"))
print("Rentang hari produksi :",
      sw.agg(F.min("production_date"), F.max("production_date")).collect()[0])
print(f"Total trip            : {sw.count():,}")
print(f"Total netto           : {sw.agg(F.sum('netto_ton')).collect()[0][0]:,.1f} ton")
display(
    sw.groupBy("flow_path", "rom_code")
      .agg(F.count("*").alias("trips"),
           F.round(F.sum("netto_ton"), 1).alias("netto_ton"),
           F.round(F.avg("netto_ton"), 2).alias("avg_payload_ton"))
      .orderBy(F.desc("netto_ton"))
)

### 3.2 `silver_urise_events` - event stream truk (ROM / Stockpile / CP)

Satu baris = satu kunjungan truk ke satu node. `node_type` diturunkan dari
`master_location`; arah gerbang (`IN`/`OUT`) dari `location` / `location_out`.
Event tanpa `master_location` (3.286 baris pada sampel, semua dari device
`BNT-CP4_5IN-RFID01`) ditandai `node_type = 'UNRESOLVED'` dan **tidak** ikut
dihitung di funnel, tapi tetap dilaporkan di data-quality.

In [0]:
bronze_urise = spark.table(tbl("bronze", "bronze_urise_transaksi"))

silver_urise = (
    bronze_urise
    .withColumn("event_id", F.col("id"))
    .withColumn("truck", F.upper(F.trim(F.col("no_lambung"))))
    .withColumn("truck_prefix", contractor_from_truck(F.col("truck")))
    .withColumn("contractor_code",
                F.coalesce(F.col("truck_prefix"), F.lit("UNKNOWN")))
    .withColumn("contractor_name",
                F.coalesce(F.trim(F.col("kontraktor")),
                           contractor_name(F.col("contractor_code"))))
    # --- klasifikasi node ---
    .withColumn("node_code",
                F.when(F.col("master_location").rlike(r"(?i)^ROM"),
                       norm_rom(F.col("master_location")))
                 .when(F.col("master_location").rlike(r"(?i)STOCKPILE"),
                       F.lit("NORTH-STOCKPILE"))
                 .when(F.col("master_location").rlike(r"(?i)^CP"),
                       norm_port(F.col("master_location")))
                 .otherwise(F.lit("UNRESOLVED")))
    .withColumn("node_type",
                F.when(F.col("node_code").startswith("ROM"), F.lit("ROM"))
                 .when(F.col("node_code").startswith("PIT"), F.lit("ROM"))
                 .when(F.col("node_code") == "NORTH-STOCKPILE", F.lit("STOCKPILE"))
                 .when(F.col("node_code").startswith("CP-"), F.lit("CP"))
                 .otherwise(F.lit("UNRESOLVED")))
    .withColumn("gate_in", F.col("location"))
    .withColumn("gate_out", F.col("location_out"))
    # --- waktu ---
    .withColumn("time_in_utc", F.col("room_in"))
    .withColumn("time_out_utc", F.col("room_out"))
    .withColumn("dwell_minutes",
                F.round((F.col("room_out").cast("long")
                         - F.col("room_in").cast("long")) / 60.0, 1))
    # --- tonase & material ---
    .withColumn("tonase_ton", to_ton("tonase"))
    .withColumn("material_code", F.col("material"))
    .withColumn("material_block", F.split(F.col("material"), r"\.").getItem(0))
    .withColumn("material_seam", F.split(F.col("material"), r"\.").getItem(1))
    .withColumn("material_quality", F.split(F.col("material"), r"\.").getItem(2))
    .withColumn("material_product", F.split(F.col("material"), r"\.").getItem(3))
    .withColumn("rom_ticket_no", F.col("sk"))            # nomor SK muat ROM
    .withColumn("loader_type", F.col("alat_muat"))       # EXCA / TLS
    .withColumn("capture_source", F.col("data_source"))  # RFID / GEOFENCE
    # --- flag ---
    .withColumn("is_stockpile_fleet",
                F.col("contractor_code").isin(*STOCKPILE_FLEET_PREFIXES))
    .withColumn("is_anomaly_flag", F.coalesce(F.col("is_anomaly") == "true", F.lit(False)))
    .withColumn("is_deleted_flag", F.coalesce(F.col("is_deleted") == "true", F.lit(False)))
    .withColumn("has_tonnage", F.col("tonase_ton").isNotNull())
    .withColumn("is_open_visit", F.col("time_out_utc").isNull())
    .withColumn("_source_system", F.lit("URISE"))
)

silver_urise = add_time_columns(silver_urise, "time_in_utc")

silver_urise = (
    silver_urise
    .select(
        "event_id", "truck", "contractor_code", "contractor_name",
        "node_code", "node_type", "gate_in", "gate_out",
        "time_in_utc", "time_out_utc", "dwell_minutes",
        "tonase_ton", "material_code", "material_block", "material_seam",
        "material_quality", "material_product", "rom_ticket_no",
        "loader_type", "capture_source", "rfid", "device_id",
        "ts_local", "date_local", "hour_local", "production_date", "shift",
        "week_start", "month",
        "is_stockpile_fleet", "is_anomaly_flag", "is_deleted_flag",
        "has_tonnage", "is_open_visit",
        "_source_system", "_ingested_at",
    )
    .filter(F.col("event_id").isNotNull())
    .dropDuplicates(["event_id"])
)

write_table(silver_urise, "silver", "silver_urise_events",
            partition_by=["production_date"],
            comment="Event stream Urise per kunjungan truk ke node (ROM/Stockpile/CP), tonase ton.")

display(
    silver_urise.groupBy("node_type", "capture_source")
    .agg(F.count("*").alias("events"),
         F.sum(F.col("has_tonnage").cast("int")).alias("with_tonnage"),
         F.round(F.sum("tonase_ton"), 1).alias("total_ton"))
    .orderBy("node_type", "capture_source")
)

#### 3.2.1 Turunan Urise per stage

Tiga view Silver yang langsung dipakai funnel. Untuk node ROM dan CP,
tonase yang dihitung hanya dari event yang **punya tonase**; jumlah event tanpa
tonase dilaporkan terpisah supaya coverage terlihat di dashboard.

In [0]:
su = spark.table(tbl("silver", "silver_urise_events"))

# --- ROM loading (produksi di ROM) ---
silver_rom = (
    su.filter((F.col("node_type") == "ROM") & (~F.col("is_deleted_flag")))
      .withColumnRenamed("node_code", "rom_code")
      .withColumn("rom_exit_ts_utc", F.coalesce(F.col("time_out_utc"), F.col("time_in_utc")))
)
write_table(silver_rom, "silver", "silver_rom_loading",
            partition_by=["production_date"],
            comment="Event pemuatan di ROM (Urise) - stage 1 funnel.")

# --- Kunjungan stockpile (Urise, untuk cross-check Uscavis) ---
silver_sp_visit = (
    su.filter((F.col("node_type") == "STOCKPILE") & (~F.col("is_deleted_flag")))
      .withColumn("direction",
                  F.when(F.coalesce(F.col("gate_in"), F.lit("")).rlike("(?i)GATE3"), F.lit("IN"))
                   .when(F.coalesce(F.col("gate_out"), F.lit("")).rlike("(?i)GATE4|GATE5"), F.lit("OUT"))
                   .otherwise(F.lit("UNKNOWN")))
)
write_table(silver_sp_visit, "silver", "silver_stockpile_visit",
            partition_by=["production_date"],
            comment="Kunjungan truk ke North Stockpile menurut Urise (cross-check Uscavis).")

# --- Dumping di Crushing Plant ---
silver_cp = (
    su.filter((F.col("node_type") == "CP") & (~F.col("is_deleted_flag")))
      .withColumnRenamed("node_code", "cp_code")
      .withColumn("dump_ts_utc", F.coalesce(F.col("time_out_utc"), F.col("time_in_utc")))
      .withColumn("source_leg",
                  F.when(F.col("is_stockpile_fleet"), F.lit("FROM_STOCKPILE"))
                   .otherwise(F.lit("FROM_ROM_DIRECT")))
)
write_table(silver_cp, "silver", "silver_cp_dumping",
            partition_by=["production_date"],
            comment="Event dumping truk di hopper Crushing Plant (Urise) - stage 5 funnel.")

print("\
Coverage tonase per stage Urise:")
display(
    su.filter(F.col("node_type") != "UNRESOLVED")
      .groupBy("node_type")
      .agg(F.count("*").alias("events"),
           F.sum(F.col("has_tonnage").cast("int")).alias("events_with_ton"),
           F.round(F.avg(F.col("has_tonnage").cast("int")) * 100, 1).alias("coverage_pct"),
           F.round(F.sum("tonase_ton"), 1).alias("total_ton"))
)

### 3.3 `silver_uscavis_stock_txn` - mutasi stok North Stockpile

`movement = IN` menambah stok, `OUT` mengurangi. `signed_ton` sudah bertanda
sehingga saldo tinggal `sum()` kumulatif. Nomor lambung truk diekstrak dari
`remarks` untuk bisa di-join ke Urise/WIM.

In [0]:
bronze_uscavis = spark.table(tbl("bronze", "bronze_uscavis_stock"))

silver_stock = (
    bronze_uscavis
    .withColumn("stock_txn_id", F.col("id"))
    .withColumn("txn_ts_utc", F.col("transaction_date"))
    .withColumn("movement", F.upper(F.trim(F.col("movement"))))
    .withColumn("qty_ton", to_ton("quantity"))
    .withColumn("signed_ton",
                F.when(F.col("movement") == "IN", F.col("qty_ton"))
                 .when(F.col("movement") == "OUT", -F.col("qty_ton"))
                 .otherwise(F.lit(0.0)))
    .withColumn("truck",
                F.upper(F.trim(F.regexp_extract(
                    F.coalesce(F.col("remarks"), F.lit("")),
                    r"(?i)for\s+([A-Za-z]+\s*[0-9]+)", 1))))
    .withColumn("truck", F.when(F.col("truck") == "", F.lit(None)).otherwise(F.col("truck")))
    .withColumn("contractor_code", contractor_from_truck(F.col("truck")))
    .withColumn("contractor_name", contractor_name(F.col("contractor_code")))
    .withColumn("stockpile_code", F.upper(F.trim(F.col("location"))))
    .withColumn("capture_source", F.col("source_system"))     # RFID / GEOFENCE
    .withColumn("is_final", F.col("status") == F.lit("FINAL"))
    .withColumn("is_zero_qty", F.col("qty_ton") <= 0)
    .withColumn("_source_system", F.lit("USCAVIS"))
)

silver_stock = add_time_columns(silver_stock, "txn_ts_utc")

silver_stock = (
    silver_stock
    .select("stock_txn_id", "txn_ts_utc", "movement", "qty_ton", "signed_ton",
            "truck", "contractor_code", "contractor_name",
            "stockpile_code", "capture_source", "status", "is_final",
            "is_zero_qty", "remarks",
            "ts_local", "date_local", "hour_local", "production_date", "shift",
            "week_start", "month", "_source_system", "_ingested_at")
    .filter(F.col("stock_txn_id").isNotNull())
    .dropDuplicates(["stock_txn_id"])
)

write_table(silver_stock, "silver", "silver_uscavis_stock_txn",
            partition_by=["production_date"],
            comment="Mutasi stok North Stockpile (Uscavis) - stage 3 & 4 funnel, satuan ton.")

display(
    silver_stock.groupBy("movement", "status")
    .agg(F.count("*").alias("txn"),
         F.round(F.sum("qty_ton"), 1).alias("total_ton"),
         F.sum(F.col("is_zero_qty").cast("int")).alias("zero_qty_rows"))
)

### 3.4 `silver_trip_reconciliation` - rekonsiliasi level trip

Inti dari use case: menyambung **satu trip fisik** lintas tiga sistem.

Aturan pencocokan (semua `LEFT JOIN`, trip WIM tetap dipertahankan meskipun tak
ada pasangan - justru trip tanpa pasangan itulah sinyal loss):

| Leg | Kunci | Aturan pemilihan |
|-----|-------|------------------|
| WIM -> event ROM Urise | `truck` sama | event ROM dengan `rom_exit_ts_utc` **terdekat sebelum** waktu timbang, maksimal `TRIP_MATCH_WINDOW_H` jam |
| WIM -> event CP Urise | `truck` sama | event CP dengan `dump_ts_utc` **paling awal setelah** waktu timbang, maksimal `TRIP_MATCH_WINDOW_H` jam |
| WIM -> mutasi Uscavis IN | `truck` sama | mutasi `IN` paling awal setelah timbang, maksimal `TRIP_MATCH_WINDOW_H` jam, hanya untuk trip `flow_path = VIA_STOCKPILE` |

Selisih dihitung terhadap `netto_ton` WIM sebagai baseline (jembatan timbang =
alat ukur paling akurat).

In [0]:
w = spark.table(tbl("silver", "silver_wim_hauling"))
rom = spark.table(tbl("silver", "silver_rom_loading"))
cp = spark.table(tbl("silver", "silver_cp_dumping"))
stk = spark.table(tbl("silver", "silver_uscavis_stock_txn"))

WIN_SEC = TRIP_MATCH_WINDOW_H * 3600

base = w.select(
    "trip_id", "truck", "contractor_code", "contractor_name", "client",
    "rom_code", "dest_code", "flow_path", "material_code", "coal_type",
    "netto_ton", "weigh_ts_utc", "rom_exit_ts_utc", "port_arrival_ts_utc",
    "rom_to_port_min", "production_date", "shift", "week_start", "month",
    "is_payload_anomaly", "is_manual_weigh",
)

# ---------- Leg 1: WIM -> event pemuatan ROM (Urise) ----------
rom_c = rom.select(
    F.col("event_id").alias("rom_event_id"),
    F.col("truck").alias("r_truck"),
    F.col("rom_code").alias("rom_code_urise"),
    F.col("rom_exit_ts_utc").alias("rom_event_ts_utc"),
    F.col("tonase_ton").alias("rom_ton_urise"),
    F.col("rom_ticket_no"),
    F.col("loader_type"),
    F.col("material_code").alias("rom_material_code"),
)

j_rom = (
    base.join(rom_c,
              (base.truck == rom_c.r_truck)
              & (rom_c.rom_event_ts_utc <= base.weigh_ts_utc)
              & (rom_c.rom_event_ts_utc >= F.expr(f"weigh_ts_utc - INTERVAL {TRIP_MATCH_WINDOW_H} HOURS")),
              "left")
    .withColumn("_gap",
                F.col("weigh_ts_utc").cast("long") - F.col("rom_event_ts_utc").cast("long"))
)
wr = Window.partitionBy("trip_id").orderBy(F.asc_nulls_last("_gap"))
j_rom = (j_rom.withColumn("_rn", F.row_number().over(wr))
             .filter(F.col("_rn") == 1)
             .drop("_rn", "_gap", "r_truck"))

# ---------- Leg 2: WIM -> event dumping di CP (Urise) ----------
cp_c = cp.select(
    F.col("event_id").alias("cp_event_id"),
    F.col("truck").alias("c_truck"),
    F.col("cp_code"),
    F.col("dump_ts_utc").alias("cp_dump_ts_utc"),
    F.col("tonase_ton").alias("cp_ton_urise"),
    F.col("source_leg").alias("cp_source_leg"),
)

j_cp = (
    j_rom.join(cp_c,
               (j_rom.truck == cp_c.c_truck)
               & (cp_c.cp_dump_ts_utc >= j_rom.weigh_ts_utc)
               & (cp_c.cp_dump_ts_utc <= F.expr(f"weigh_ts_utc + INTERVAL {TRIP_MATCH_WINDOW_H} HOURS")),
               "left")
    .withColumn("_gap",
                F.col("cp_dump_ts_utc").cast("long") - F.col("weigh_ts_utc").cast("long"))
)
wc = Window.partitionBy("trip_id").orderBy(F.asc_nulls_last("_gap"))
j_cp = (j_cp.withColumn("_rn", F.row_number().over(wc))
            .filter(F.col("_rn") == 1)
            .drop("_rn", "_gap", "c_truck"))

# ---------- Leg 3: WIM -> mutasi masuk stockpile (Uscavis) ----------
stk_in = (
    stk.filter(F.col("movement") == "IN")
       .select(F.col("stock_txn_id").alias("stock_in_txn_id"),
               F.col("truck").alias("s_truck"),
               F.col("txn_ts_utc").alias("stock_in_ts_utc"),
               F.col("qty_ton").alias("stock_in_ton"))
)

j_stk = (
    j_cp.join(stk_in,
              (j_cp.truck == stk_in.s_truck)
              & (j_cp.flow_path == F.lit("VIA_STOCKPILE"))
              & (stk_in.stock_in_ts_utc >= j_cp.weigh_ts_utc)
              & (stk_in.stock_in_ts_utc <= F.expr(f"weigh_ts_utc + INTERVAL {TRIP_MATCH_WINDOW_H} HOURS")),
              "left")
    .withColumn("_gap",
                F.col("stock_in_ts_utc").cast("long") - F.col("weigh_ts_utc").cast("long"))
)
ws = Window.partitionBy("trip_id").orderBy(F.asc_nulls_last("_gap"))
j_stk = (j_stk.withColumn("_rn", F.row_number().over(ws))
              .filter(F.col("_rn") == 1)
              .drop("_rn", "_gap", "s_truck"))

# ---------- Selisih & flag ----------
silver_recon = (
    j_stk
    .withColumn("delta_rom_vs_wim_ton",
                F.round(F.col("rom_ton_urise") - F.col("netto_ton"), 3))
    .withColumn("delta_wim_vs_cp_ton",
                F.round(F.col("netto_ton") - F.col("cp_ton_urise"), 3))
    .withColumn("delta_rom_vs_cp_ton",
                F.round(F.col("rom_ton_urise") - F.col("cp_ton_urise"), 3))
    .withColumn("loss_pct_wim_vs_cp",
                F.round(F.when(F.col("netto_ton") > 0,
                               (F.col("netto_ton") - F.col("cp_ton_urise"))
                               / F.col("netto_ton") * 100.0), 2))
    .withColumn("rom_code_mismatch",
                F.col("rom_code_urise").isNotNull()
                & (F.col("rom_code_urise") != F.col("rom_code")))
    .withColumn("is_matched_rom", F.col("rom_event_id").isNotNull())
    .withColumn("is_matched_cp", F.col("cp_event_id").isNotNull())
    .withColumn("is_matched_stock", F.col("stock_in_txn_id").isNotNull())
    .withColumn("match_completeness",
                (F.col("is_matched_rom").cast("int")
                 + F.col("is_matched_cp").cast("int")))
    .withColumn("trip_status",
                F.when(F.col("is_matched_rom") & F.col("is_matched_cp"), F.lit("FULL_TRACE"))
                 .when(F.col("is_matched_cp"), F.lit("CP_ONLY"))
                 .when(F.col("is_matched_rom"), F.lit("ROM_ONLY"))
                 .otherwise(F.lit("WEIGH_ONLY")))
    .withColumn("cycle_time_min",
                F.round((F.col("cp_dump_ts_utc").cast("long")
                         - F.col("rom_event_ts_utc").cast("long")) / 60.0, 1))
    .withColumn("_reconciled_at", F.current_timestamp())
)

write_table(silver_recon, "silver", "silver_trip_reconciliation",
            partition_by=["production_date"],
            comment="Rekonsiliasi level trip: WIM <-> Urise ROM <-> Urise CP <-> Uscavis stock IN.")

display(
    silver_recon.groupBy("trip_status")
    .agg(F.count("*").alias("trips"),
         F.round(F.sum("netto_ton"), 1).alias("netto_ton"),
         F.round(F.avg("loss_pct_wim_vs_cp"), 2).alias("avg_loss_pct"))
    .orderBy(F.desc("trips"))
)

---
## 4. GOLD LAYER - Mart untuk dashboard

Semua tabel gold memakai grain yang eksplisit dan sudah dalam satuan **ton**,
siap dipakai langsung oleh Databricks SQL / Lakeview tanpa join tambahan yang berat.

| Tabel gold | Grain | Visual yang dilayani |
|---|---|---|
| `gold_funnel_stage_daily` | tanggal x stage | Funnel ROM -> Hauling -> Stockpile -> CP -> Shipment |
| `gold_stage_reconciliation` | tanggal x pasangan stage | Selisih volumetrik & % loss/recovery per tahap |
| `gold_production_timeseries` | grain x periode | Time-series harian/mingguan/bulanan vs target |
| `gold_contractor_daily` | tanggal x kontraktor | Balancing kontraktor, ritase, payload |
| `gold_rom_daily` | tanggal x ROM | Produksi per ROM vs target |
| `gold_cp_intake_daily` | tanggal x CP | Intake per Crushing Plant, direct vs reclaim |
| `gold_stockpile_balance_hourly` | jam | Saldo North Stockpile & days-of-cover |
| `gold_loss_anomaly` | tanggal x dimensi | Deteksi loss tidak wajar (robust z-score) |
| `gold_shipment_outlook` | bulan | Prediksi shortfall shipment |
| `gold_suggestion_daily` | tanggal x rank | Top-3 rekomendasi harian (Mosaic AI) |
| `gold_kpi_snapshot` | tanggal | Kartu KPI headline |

### 4.1 Sumber shipment (stage 7) - dari gold UC-06

Tonase shipment tidak ada di WIM/Urise/Uscavis, tapi **ada di UC-06**:
`uc.uscavis.gold_wim_cv_ratio.conveyor_tonnage` = total tonase conveyor BLC per hari,
yaitu gabungan jalur *direct* (CP -> conveyor -> shipment) dan *non-direct*
(stockpile -> reclaim -> conveyor -> shipment). Itu angka aktual, bukan estimasi.

Urutan sumber yang dicoba (`SHIPMENT_SOURCES`):

1. `uc.uscavis.gold_wim_cv_ratio` kolom `conveyor_tonnage` (utama),
2. `uc.uscavis.gold_output_summary` kolom `total_tonnage_ton` (cadangan; ini
   sebenarnya output totalizer CP, jadi lebih dekat ke stage 6 daripada stage 7),
3. estimasi `CP intake x CRUSHING_RECOVERY_FACTOR` - **selalu** ditandai
   `is_estimated = true`.

Catatan satuan: UC-06 sudah mengalikan volume conveyor dengan density BIB
`0.88`, jadi `conveyor_tonnage` sudah dalam ton dan tidak perlu dikonversi lagi.

In [0]:
cp_daily_ton = (
    spark.table(tbl("silver", "silver_cp_dumping"))
    .filter(F.col("has_tonnage"))
    .groupBy("production_date")
    .agg(F.sum("tonase_ton").alias("cp_intake_ton"))
)


def load_shipment_daily() -> DataFrame:
    """Ambil tonase shipment harian dari gold UC-06; fallback ke estimasi."""
    for src_cfg in SHIPMENT_SOURCES:
        fqn, dcol, tcol = src_cfg["table"], src_cfg["date_col"], src_cfg["ton_col"]
        try:
            src = spark.table(fqn)
        except Exception as e:                              # noqa: BLE001
            print(f"[SKIP] {fqn} tidak bisa dibaca ({type(e).__name__}).")
            continue
        if dcol not in src.columns or tcol not in src.columns:
            print(f"[SKIP] {fqn}: kolom {dcol}/{tcol} tidak ada "
                  f"(kolom tersedia: {src.columns[:8]}...).")
            continue
        out = (src.select(F.col(dcol).cast("date").alias("production_date"),
                          F.col(tcol).cast("double").alias("shipment_ton"))
                  .filter(F.col("production_date").isNotNull())
                  .groupBy("production_date")
                  .agg(F.round(F.sum("shipment_ton"), 3).alias("shipment_ton"))
                  .withColumn("is_estimated", F.lit(False))
                  .withColumn("shipment_source", F.lit(f"{fqn}.{tcol}")))
        print(f"Shipment dibaca dari {fqn}.{tcol} "
              f"({src_cfg['label']}): {out.count():,} hari")
        return out

    print("[WARN] Semua sumber shipment UC-06 tidak tersedia.")
    if not SHIPMENT_FALLBACK_ENABLED:
        print("       SHIPMENT_FALLBACK_ENABLED=False -> stage SHIPMENT dikosongkan.")
        return (cp_daily_ton.limit(0)
                .select(F.col("production_date"),
                        F.lit(None).cast("double").alias("shipment_ton"),
                        F.lit(True).alias("is_estimated"),
                        F.lit("NONE").alias("shipment_source")))
    print(f"       Fallback: CP intake x {CRUSHING_RECOVERY_FACTOR} (ditandai is_estimated).")
    return (cp_daily_ton
            .withColumn("shipment_ton",
                        F.round(F.col("cp_intake_ton") * F.lit(CRUSHING_RECOVERY_FACTOR), 3))
            .withColumn("is_estimated", F.lit(True))
            .withColumn("shipment_source",
                        F.lit(f"ESTIMATED = cp_intake x {CRUSHING_RECOVERY_FACTOR}"))
            .select("production_date", "shipment_ton", "is_estimated", "shipment_source"))


shipment_daily = load_shipment_daily()
display(shipment_daily.orderBy("production_date"))

### 4.2 `gold_funnel_stage_daily`

Satu baris per (hari produksi, stage). Kolom `tonnage_ton` adalah angka funnel,
`trips` jumlah ritase, `source_system` menjelaskan asal angka (penting untuk audit:
tiap tahap diukur alat yang berbeda).

In [0]:
sw = spark.table(tbl("silver", "silver_wim_hauling"))
srom = spark.table(tbl("silver", "silver_rom_loading"))
scp = spark.table(tbl("silver", "silver_cp_dumping"))
sstk = spark.table(tbl("silver", "silver_uscavis_stock_txn"))


def stage_row(df, seq, code_, label, branch, source, ton_col, trip_col=None):
    agg = df.groupBy("production_date").agg(
        F.round(F.sum(ton_col), 3).alias("tonnage_ton"),
        F.count(F.lit(1)).alias("trips"),
        F.countDistinct(trip_col).alias("distinct_units") if trip_col
        else F.lit(None).cast("long").alias("distinct_units"),
    )
    return (agg
            .withColumn("stage_seq", F.lit(seq))
            .withColumn("stage_code", F.lit(code_))
            .withColumn("stage_name", F.lit(label))
            .withColumn("stage_branch", F.lit(branch))
            .withColumn("source_system", F.lit(source))
            .select("production_date", "stage_seq", "stage_code", "stage_name",
                    "stage_branch", "source_system", "tonnage_ton", "trips",
                    "distinct_units"))


parts = [
    stage_row(srom.filter(F.col("has_tonnage")), 1, "ROM_PRODUCTION",
              "ROM Production", "MAIN", "URISE", "tonase_ton", "truck"),
    stage_row(sw, 2, "HAULING_WEIGHED",
              "Hauling (WIM/Ucan)", "MAIN", "WIM_UCAN", "netto_ton", "truck"),
    stage_row(sw.filter(F.col("flow_path") == "DIRECT_CP"), 3, "HAULING_DIRECT_CP",
              "Direct ROM -> CP", "DIRECT", "WIM_UCAN", "netto_ton", "truck"),
    stage_row(sstk.filter(F.col("movement") == "IN"), 4, "STOCKPILE_IN",
              "Stockpile IN", "BUFFER", "USCAVIS", "qty_ton", "truck"),
    stage_row(sstk.filter(F.col("movement") == "OUT"), 5, "STOCKPILE_OUT",
              "Stockpile OUT/Reclaim", "BUFFER", "USCAVIS", "qty_ton", "truck"),
    stage_row(scp.filter(F.col("has_tonnage")), 6, "CP_INTAKE",
              "Crushing Plant Intake", "MAIN", "URISE", "tonase_ton", "truck"),
]

funnel = parts[0]
for p in parts[1:]:
    funnel = funnel.unionByName(p)

# Stage 7 - shipment (bisa estimasi)
ship_stage = (
    shipment_daily
    .select("production_date",
            F.lit(7).alias("stage_seq"),
            F.lit("SHIPMENT").alias("stage_code"),
            F.lit("Shipment").alias("stage_name"),
            F.lit("MAIN").alias("stage_branch"),
            F.col("shipment_source").alias("source_system"),
            F.round(F.col("shipment_ton"), 3).alias("tonnage_ton"),
            F.lit(None).cast("long").alias("trips"),
            F.lit(None).cast("long").alias("distinct_units"))
)
funnel = funnel.unionByName(ship_stage)

# Baseline funnel = HAULING_WEIGHED (jembatan timbang = pengukuran paling akurat)
baseline = (funnel.filter(F.col("stage_code") == "HAULING_WEIGHED")
                  .select("production_date",
                          F.col("tonnage_ton").alias("baseline_ton")))

ship_flag = shipment_daily.select("production_date",
                                  F.col("is_estimated").alias("_ship_est"))

gold_funnel = (
    funnel.join(baseline, "production_date", "left")
    .join(ship_flag, "production_date", "left")
    .withColumn("pct_of_baseline", safe_pct("tonnage_ton", "baseline_ton"))
    # hanya stage SHIPMENT yang bisa berisi estimasi; stage lain selalu terukur
    .withColumn("is_estimated",
                F.when(F.col("stage_code") == F.lit("SHIPMENT"),
                       F.coalesce(F.col("_ship_est"), F.lit(True)))
                 .otherwise(F.lit(False)))
    .drop("_ship_est")
    .withColumn("_generated_at", F.current_timestamp())
    .orderBy("production_date", "stage_seq")
)

write_table(gold_funnel, "gold", "gold_funnel_stage_daily",
            partition_by=["production_date"],
            comment="Funnel produksi harian per stage (ROM -> Hauling -> Stockpile -> CP -> Shipment).")

display(gold_funnel.orderBy("production_date", "stage_seq"))

### 4.3 `gold_stage_reconciliation` - selisih volumetrik & % loss/recovery

Pasangan tahap yang direkonsiliasi:

| Pasangan | Arti selisih |
|---|---|
| `ROM_PRODUCTION -> HAULING_WEIGHED` | selisih pengukuran alat muat vs jembatan timbang |
| `HAULING_WEIGHED -> CP_INTAKE` | loss di jalan + tercecer + belum dumping |
| `STOCKPILE_IN -> STOCKPILE_OUT` | perubahan buffer stockpile (bukan loss) |
| `CP_INTAKE -> SHIPMENT` | recovery crushing + coal yang masih di plant |

`loss_pct` positif = tonase menyusut dari tahap hulu ke hilir.
`recovery_pct = 100 - loss_pct`.

In [0]:
STAGE_PAIRS = [
    ("ROM_PRODUCTION",  "HAULING_WEIGHED", "Muat ROM vs Timbang",     "MEASUREMENT_GAP"),
    ("HAULING_WEIGHED", "CP_INTAKE",       "Timbang vs Intake CP",    "TRANSPORT_LOSS"),
    ("STOCKPILE_IN",    "STOCKPILE_OUT",   "Stockpile IN vs OUT",     "BUFFER_CHANGE"),
    ("CP_INTAKE",       "SHIPMENT",        "Intake CP vs Shipment",   "PROCESS_RECOVERY"),
]

pivot = (
    spark.table(tbl("gold", "gold_funnel_stage_daily"))
    .groupBy("production_date")
    .pivot("stage_code", [s[1] for s in FUNNEL_STAGES])
    .agg(F.first("tonnage_ton"))
)

recon_parts = []
for upstream, downstream, label, kind in STAGE_PAIRS:
    recon_parts.append(
        pivot.select(
            F.col("production_date"),
            F.lit(upstream).alias("upstream_stage"),
            F.lit(downstream).alias("downstream_stage"),
            F.lit(label).alias("pair_label"),
            F.lit(kind).alias("gap_type"),
            F.round(F.col(upstream), 3).alias("upstream_ton"),
            F.round(F.col(downstream), 3).alias("downstream_ton"),
            F.round(F.col(upstream) - F.col(downstream), 3).alias("delta_ton"),
        )
    )

gold_recon = recon_parts[0]
for p in recon_parts[1:]:
    gold_recon = gold_recon.unionByName(p)

gold_recon = (
    gold_recon
    .withColumn("loss_pct",
                F.round(F.when(F.col("upstream_ton") > 0,
                               F.col("delta_ton") / F.col("upstream_ton") * 100.0), 2))
    .withColumn("recovery_pct",
                F.round(F.when(F.col("upstream_ton") > 0,
                               F.col("downstream_ton") / F.col("upstream_ton") * 100.0), 2))
    .withColumn("severity",
                F.when(F.col("gap_type") == "BUFFER_CHANGE", F.lit("INFO"))
                 .when(F.abs(F.col("loss_pct")) >= LOSS_PCT_CRITICAL, F.lit("CRITICAL"))
                 .when(F.abs(F.col("loss_pct")) >= LOSS_PCT_WARN, F.lit("WARNING"))
                 .when(F.col("loss_pct").isNull(), F.lit("NO_DATA"))
                 .otherwise(F.lit("NORMAL")))
    .withColumn("direction",
                F.when(F.col("delta_ton") > 0, F.lit("LOSS"))
                 .when(F.col("delta_ton") < 0, F.lit("GAIN"))
                 .otherwise(F.lit("BALANCED")))
    .withColumn("_generated_at", F.current_timestamp())
)

write_table(gold_recon, "gold", "gold_stage_reconciliation",
            partition_by=["production_date"],
            comment="Selisih volumetrik antar tahap funnel + persentase loss/recovery.")

display(gold_recon.orderBy("production_date", "upstream_stage"))

### 4.4 `gold_production_timeseries` - harian / mingguan / bulanan vs target

Target diambil dari `TARGET_BY_MONTH` (fallback `MONTHLY_TARGET_TON`), lalu
di-*prorate* ke grain harian dan mingguan berdasarkan jumlah hari dalam bulan.
Aktual memakai `HAULING_WEIGHED` (jembatan timbang) sebagai angka produksi resmi.

In [0]:
TARGET_MAP_EXPR = F.create_map(
    [x for kv in TARGET_BY_MONTH.items() for x in (F.lit(kv[0]), F.lit(float(kv[1])))]
) if TARGET_BY_MONTH else None


def monthly_target(month_col):
    base = F.lit(float(MONTHLY_TARGET_TON))
    return F.coalesce(TARGET_MAP_EXPR[month_col], base) if TARGET_MAP_EXPR is not None else base


wim = spark.table(tbl("silver", "silver_wim_hauling"))

daily = (
    wim.groupBy("production_date", "month", "week_start")
    .agg(F.round(F.sum("netto_ton"), 3).alias("actual_ton"),
         F.count(F.lit(1)).alias("trips"),
         F.countDistinct("truck").alias("active_trucks"),
         F.countDistinct("contractor_code").alias("active_contractors"),
         F.round(F.avg("netto_ton"), 3).alias("avg_payload_ton"))
    .withColumn("days_in_month",
                F.dayofmonth(F.last_day(F.to_date(F.concat(F.col("month"), F.lit("-01"))))))
    .withColumn("month_target_ton", monthly_target(F.col("month")))
    .withColumn("target_ton",
                F.round(F.col("month_target_ton") / F.col("days_in_month"), 3))
)

ts_daily = (
    daily.select(
        F.lit("DAILY").alias("grain"),
        F.col("production_date").alias("period_start"),
        F.col("production_date").alias("period_end"),
        F.date_format("production_date", "yyyy-MM-dd").alias("period_label"),
        "actual_ton", "target_ton", "trips", "active_trucks",
        "active_contractors", "avg_payload_ton")
)

ts_weekly = (
    daily.groupBy("week_start")
    .agg(F.round(F.sum("actual_ton"), 3).alias("actual_ton"),
         F.round(F.sum("target_ton"), 3).alias("target_ton"),
         F.sum("trips").alias("trips"),
         F.round(F.avg("active_trucks"), 1).alias("active_trucks"),
         F.round(F.avg("active_contractors"), 1).alias("active_contractors"),
         F.round(F.avg("avg_payload_ton"), 3).alias("avg_payload_ton"))
    .select(
        F.lit("WEEKLY").alias("grain"),
        F.col("week_start").alias("period_start"),
        F.date_add(F.col("week_start"), 6).alias("period_end"),
        F.concat(F.lit("W"), F.weekofyear(F.col("week_start")).cast("string"),
                 F.lit("-"), F.year(F.col("week_start")).cast("string")).alias("period_label"),
        "actual_ton", "target_ton", "trips", "active_trucks",
        "active_contractors", "avg_payload_ton")
)

ts_monthly = (
    daily.groupBy("month")
    .agg(F.round(F.sum("actual_ton"), 3).alias("actual_ton"),
         F.round(F.first("month_target_ton"), 3).alias("target_ton"),
         F.sum("trips").alias("trips"),
         F.round(F.avg("active_trucks"), 1).alias("active_trucks"),
         F.round(F.avg("active_contractors"), 1).alias("active_contractors"),
         F.round(F.avg("avg_payload_ton"), 3).alias("avg_payload_ton"))
    .select(
        F.lit("MONTHLY").alias("grain"),
        F.to_date(F.concat(F.col("month"), F.lit("-01"))).alias("period_start"),
        F.last_day(F.to_date(F.concat(F.col("month"), F.lit("-01")))).alias("period_end"),
        F.col("month").alias("period_label"),
        "actual_ton", "target_ton", "trips", "active_trucks",
        "active_contractors", "avg_payload_ton")
)

gold_ts = (
    ts_daily.unionByName(ts_weekly).unionByName(ts_monthly)
    .withColumn("variance_ton", F.round(F.col("actual_ton") - F.col("target_ton"), 3))
    .withColumn("achievement_pct", safe_pct("actual_ton", "target_ton"))
    .withColumn("status",
                F.when(F.col("achievement_pct") >= 100, F.lit("ON_TARGET"))
                 .when(F.col("achievement_pct") >= 90, F.lit("AT_RISK"))
                 .otherwise(F.lit("BEHIND")))
    .withColumn("_generated_at", F.current_timestamp())
)

# Moving average 7 periode untuk grain harian (dipakai garis tren di dashboard)
w7 = Window.partitionBy("grain").orderBy("period_start").rowsBetween(-6, 0)
gold_ts = gold_ts.withColumn(
    "ma7_actual_ton",
    F.when(F.col("grain") == "DAILY", F.round(F.avg("actual_ton").over(w7), 3)))

write_table(gold_ts, "gold", "gold_production_timeseries",
            partition_by=["grain"],
            comment="Time-series produksi harian/mingguan/bulanan vs target + MA7.")

display(gold_ts.filter(F.col("grain") == "DAILY").orderBy("period_start"))

### 4.5 `gold_contractor_daily` - performa & balancing kontraktor

Metrik yang dipakai suggestion engine untuk rekomendasi balancing:
`share_pct` (kontribusi tonase), `deviation_vs_fleet_pct` (payload rata-rata vs fleet),
`trips_per_truck` (produktivitas unit), dan `loss_pct` (dari rekonsiliasi trip).

In [0]:
recon = spark.table(tbl("silver", "silver_trip_reconciliation"))

contractor = (
    wim.groupBy("production_date", "shift", "contractor_code", "contractor_name")
    .agg(F.round(F.sum("netto_ton"), 3).alias("netto_ton"),
         F.count(F.lit(1)).alias("trips"),
         F.countDistinct("truck").alias("active_trucks"),
         F.round(F.avg("netto_ton"), 3).alias("avg_payload_ton"),
         F.round(F.expr("percentile_approx(netto_ton, 0.5)"), 3).alias("median_payload_ton"),
         F.round(F.avg("rom_to_port_min"), 1).alias("avg_rom_to_port_min"),
         F.sum(F.col("is_payload_anomaly").cast("int")).alias("payload_anomaly_trips"),
         F.countDistinct("rom_code").alias("rom_served"),
         F.countDistinct("dest_code").alias("dest_served"))
    .withColumn("trips_per_truck",
                F.round(F.col("trips") / F.col("active_trucks"), 2))
)

loss_by_contractor = (
    recon.filter(F.col("is_matched_cp"))
    .groupBy("production_date", "shift", "contractor_code")
    .agg(F.round(F.avg("loss_pct_wim_vs_cp"), 2).alias("avg_loss_pct"),
         F.round(F.sum("delta_wim_vs_cp_ton"), 3).alias("total_loss_ton"),
         F.count(F.lit(1)).alias("traced_trips"))
)

wday = Window.partitionBy("production_date")
gold_contractor = (
    contractor.join(loss_by_contractor,
                    ["production_date", "shift", "contractor_code"], "left")
    .withColumn("day_total_ton", F.sum("netto_ton").over(wday))
    .withColumn("share_pct", safe_pct("netto_ton", "day_total_ton"))
    .withColumn("fleet_avg_payload_ton", F.round(F.avg("avg_payload_ton").over(wday), 3))
    .withColumn("deviation_vs_fleet_pct",
                F.round((F.col("avg_payload_ton") - F.col("fleet_avg_payload_ton"))
                        / F.col("fleet_avg_payload_ton") * 100.0, 2))
    .withColumn("fleet_avg_trips_per_truck",
                F.round(F.avg("trips_per_truck").over(wday), 2))
    .withColumn("productivity_gap_pct",
                F.round((F.col("trips_per_truck") - F.col("fleet_avg_trips_per_truck"))
                        / F.col("fleet_avg_trips_per_truck") * 100.0, 2))
    .withColumn("rank_by_ton",
                F.row_number().over(Window.partitionBy("production_date")
                                    .orderBy(F.desc("netto_ton"))))
    .withColumn("balancing_flag",
                F.when(F.col("productivity_gap_pct") <= -CONTRACTOR_IMBALANCE_PCT,
                       F.lit("UNDER_UTILISED"))
                 .when(F.col("productivity_gap_pct") >= CONTRACTOR_IMBALANCE_PCT,
                       F.lit("OVER_UTILISED"))
                 .otherwise(F.lit("BALANCED")))
    .withColumn("_generated_at", F.current_timestamp())
)

write_table(gold_contractor, "gold", "gold_contractor_daily",
            partition_by=["production_date"],
            comment="Performa kontraktor hauling per hari & shift; input balancing suggestion engine.")

display(gold_contractor.orderBy(F.desc("production_date"), "rank_by_ton"))

### 4.6 `gold_rom_daily` dan `gold_cp_intake_daily`

In [0]:
# ---------- Produksi per ROM ----------
ROM_SHARE_EXPR = F.create_map(
    [x for kv in ROM_TARGET_SHARE.items() for x in (F.lit(kv[0]), F.lit(float(kv[1])))]
)

rom_urise = (
    spark.table(tbl("silver", "silver_rom_loading"))
    .filter(F.col("has_tonnage"))
    .groupBy("production_date", F.col("rom_code"))
    .agg(F.round(F.sum("tonase_ton"), 3).alias("rom_loaded_ton_urise"),
         F.count(F.lit(1)).alias("rom_load_events"),
         F.countDistinct("rom_ticket_no").alias("rom_tickets"),
         F.countDistinct("loader_type").alias("loader_types"))
)

gold_rom = (
    wim.groupBy("production_date", "month", "rom_code")
    .agg(F.round(F.sum("netto_ton"), 3).alias("hauled_ton"),
         F.count(F.lit(1)).alias("trips"),
         F.countDistinct("truck").alias("active_trucks"),
         F.countDistinct("contractor_code").alias("contractors"),
         F.round(F.avg("netto_ton"), 3).alias("avg_payload_ton"),
         F.round(F.avg("rom_to_port_min"), 1).alias("avg_haul_time_min"),
         F.countDistinct("material_code").alias("material_variants"),
         F.first("client").alias("primary_client"))
    .join(rom_urise, ["production_date", "rom_code"], "left")
    .withColumn("days_in_month",
                F.dayofmonth(F.last_day(F.to_date(F.concat(F.col("month"), F.lit("-01"))))))
    .withColumn("rom_share", F.coalesce(ROM_SHARE_EXPR[F.col("rom_code")], F.lit(0.0)))
    .withColumn("target_ton",
                F.round(monthly_target(F.col("month")) * F.col("rom_share")
                        / F.col("days_in_month"), 3))
    .withColumn("achievement_pct", safe_pct("hauled_ton", "target_ton"))
    .withColumn("delta_urise_vs_wim_ton",
                F.round(F.col("rom_loaded_ton_urise") - F.col("hauled_ton"), 3))
    .withColumn("delta_urise_vs_wim_pct", safe_pct("delta_urise_vs_wim_ton", "hauled_ton"))
    .withColumn("_generated_at", F.current_timestamp())
)

write_table(gold_rom, "gold", "gold_rom_daily",
            partition_by=["production_date"],
            comment="Produksi harian per ROM (WIM) + cross-check tonase muat Urise vs target.")

display(gold_rom.orderBy(F.desc("production_date"), F.desc("hauled_ton")))

In [0]:
# ---------- Intake per Crushing Plant ----------
cp_evt = spark.table(tbl("silver", "silver_cp_dumping"))

cp_from_urise = (
    cp_evt.groupBy("production_date", "shift", "cp_code")
    .agg(F.round(F.sum("tonase_ton"), 3).alias("intake_ton"),
         F.count(F.lit(1)).alias("dump_events"),
         F.sum(F.col("has_tonnage").cast("int")).alias("events_with_ton"),
         F.countDistinct("truck").alias("trucks"),
         F.round(F.sum(F.when(F.col("source_leg") == "FROM_STOCKPILE",
                              F.col("tonase_ton")).otherwise(0.0)), 3).alias("reclaim_ton"),
         F.round(F.sum(F.when(F.col("source_leg") == "FROM_ROM_DIRECT",
                              F.col("tonase_ton")).otherwise(0.0)), 3).alias("direct_ton"),
         F.round(F.avg("dwell_minutes"), 1).alias("avg_dwell_min"),
         F.countDistinct("hour_local").alias("active_hours"))
)

# Sisi WIM: trip yang tujuannya CP tertentu (cross-check intake)
cp_from_wim = (
    wim.filter(F.col("dest_code").startswith("CP-"))
    .groupBy("production_date", "shift", F.col("dest_code").alias("cp_code"))
    .agg(F.round(F.sum("netto_ton"), 3).alias("wim_destined_ton"),
         F.count(F.lit(1)).alias("wim_trips"))
)

# Cross-check ke gold UC-06: output totalizer CP per shift.
# UC-06 memakai penamaan 'CP1'/'CP2A' dan kolom date/shift/cp_name.
def load_uc06_cp_output() -> Optional[DataFrame]:
    try:
        src = spark.table(UC06_CP_OUTPUT_TABLE)
    except Exception as e:                                  # noqa: BLE001
        print(f"[SKIP] {UC06_CP_OUTPUT_TABLE} tidak tersedia ({type(e).__name__}); "
              f"kolom cp_output_ton dikosongkan.")
        return None
    need = {"date", "shift", "cp_name", "total_tonnage_ton"}
    if not need.issubset(set(src.columns)):
        print(f"[SKIP] {UC06_CP_OUTPUT_TABLE}: kolom kurang ({need - set(src.columns)}).")
        return None
    print(f"Cross-check UC-06 dari {UC06_CP_OUTPUT_TABLE}")
    return (src.groupBy(F.col("date").cast("date").alias("production_date"),
                        F.col("shift").alias("shift"),
                        F.upper(F.trim(F.col("cp_name"))).alias("cp_name_uc06"))
               .agg(F.round(F.sum("total_tonnage_ton"), 3).alias("cp_output_ton"),
                    F.round(F.avg("running_hours"), 2).alias("cp_running_hours"),
                    F.round(F.avg("productivity_tph"), 2).alias("cp_productivity_tph")))


uc06_cp = load_uc06_cp_output()

wcp = Window.partitionBy("production_date")
gold_cp = (
    cp_from_urise.join(cp_from_wim, ["production_date", "shift", "cp_code"], "left")
    .withColumn("cp_name_uc06", cp_name_uc06(F.col("cp_code")))
    .withColumn("ton_coverage_pct", safe_pct("events_with_ton", "dump_events"))
    .withColumn("delta_wim_vs_cp_ton",
                F.round(F.col("wim_destined_ton") - F.col("intake_ton"), 3))
    .withColumn("delta_wim_vs_cp_pct", safe_pct("delta_wim_vs_cp_ton", "wim_destined_ton"))
    .withColumn("day_total_intake_ton", F.sum("intake_ton").over(wcp))
    .withColumn("share_pct", safe_pct("intake_ton", "day_total_intake_ton"))
    .withColumn("throughput_tph",
                F.round(F.col("intake_ton") / F.greatest(F.col("active_hours"), F.lit(1)), 2))
    .withColumn("rank_by_intake",
                F.row_number().over(Window.partitionBy("production_date", "shift")
                                    .orderBy(F.desc("intake_ton"))))
    .withColumn("_generated_at", F.current_timestamp())
)

# Intake (masuk hopper, dari Urise) vs output (keluar totalizer CP, dari UC-06).
# Selisihnya = coal yang masih di dalam plant + susut proses crushing.
if uc06_cp is not None:
    gold_cp = (
        gold_cp.join(uc06_cp, ["production_date", "shift", "cp_name_uc06"], "left")
        .withColumn("intake_vs_output_ton",
                    F.round(F.col("intake_ton") - F.col("cp_output_ton"), 3))
        .withColumn("crushing_recovery_pct", safe_pct("cp_output_ton", "intake_ton"))
    )
else:
    gold_cp = (
        gold_cp
        .withColumn("cp_output_ton", F.lit(None).cast("double"))
        .withColumn("cp_running_hours", F.lit(None).cast("double"))
        .withColumn("cp_productivity_tph", F.lit(None).cast("double"))
        .withColumn("intake_vs_output_ton", F.lit(None).cast("double"))
        .withColumn("crushing_recovery_pct", F.lit(None).cast("double"))
    )

write_table(gold_cp, "gold", "gold_cp_intake_daily",
            partition_by=["production_date"],
            comment="Intake harian per Crushing Plant (direct vs reclaim) + cross-check "
                    "output totalizer CP dari UC-06 gold_output_summary.")

display(gold_cp.orderBy(F.desc("production_date"), "cp_code"))

### 4.7 `gold_stockpile_balance_hourly` - saldo North Stockpile

Saldo kumulatif = `OPENING_STOCK_TON` + cumulative sum `signed_ton`. Bila stock
opname belum tersedia, `OPENING_STOCK_TON = 0` sehingga angkanya adalah **saldo
relatif** (perubahan sejak awal window) - berguna untuk melihat tren buffer,
tetapi bukan absolut. `days_of_cover` = saldo / rata-rata konsumsi CP harian.

In [0]:
stk = spark.table(tbl("silver", "silver_uscavis_stock_txn"))

hourly = (
    stk.withColumn("hour_start_utc", F.date_trunc("hour", F.col("txn_ts_utc")))
    .groupBy("hour_start_utc", "production_date", "shift", "date_local", "hour_local")
    .agg(F.round(F.sum(F.when(F.col("movement") == "IN", F.col("qty_ton"))
                        .otherwise(0.0)), 3).alias("in_ton"),
         F.round(F.sum(F.when(F.col("movement") == "OUT", F.col("qty_ton"))
                        .otherwise(0.0)), 3).alias("out_ton"),
         F.round(F.sum("signed_ton"), 3).alias("net_ton"),
         F.sum(F.when(F.col("movement") == "IN", 1).otherwise(0)).alias("in_txn"),
         F.sum(F.when(F.col("movement") == "OUT", 1).otherwise(0)).alias("out_txn"),
         F.countDistinct("truck").alias("distinct_trucks"))
)

wcum = Window.orderBy("hour_start_utc").rowsBetween(Window.unboundedPreceding, 0)

avg_daily_cp = (
    spark.table(tbl("gold", "gold_cp_intake_daily"))
    .groupBy("production_date").agg(F.sum("intake_ton").alias("d"))
    .agg(F.avg("d")).collect()[0][0] or 0.0
)
print(f"Rata-rata konsumsi CP harian: {avg_daily_cp:,.1f} ton")

gold_stock = (
    hourly
    .withColumn("balance_ton",
                F.round(F.lit(float(OPENING_STOCK_TON)) + F.sum("net_ton").over(wcum), 3))
    .withColumn("avg_daily_cp_ton", F.lit(round(float(avg_daily_cp), 3)))
    .withColumn("days_of_cover",
                F.round(F.when(F.lit(avg_daily_cp) > 0,
                               F.col("balance_ton") / F.lit(float(avg_daily_cp))), 2))
    .withColumn("buffer_status",
                F.when(F.col("days_of_cover").isNull(), F.lit("UNKNOWN"))
                 .when(F.col("days_of_cover") < STOCKPILE_MIN_COVER_DAYS, F.lit("LOW"))
                 .when(F.col("days_of_cover") < STOCKPILE_MIN_COVER_DAYS * 2, F.lit("ADEQUATE"))
                 .otherwise(F.lit("HEALTHY")))
    .withColumn("is_relative_balance", F.lit(float(OPENING_STOCK_TON) == 0.0))
    .withColumn("_generated_at", F.current_timestamp())
)

write_table(gold_stock, "gold", "gold_stockpile_balance_hourly",
            partition_by=["production_date"],
            comment="Saldo North Stockpile per jam (IN/OUT/net/kumulatif) + days-of-cover.")

display(gold_stock.orderBy("hour_start_utc"))

### 4.8 `gold_loss_anomaly` - deteksi loss tidak wajar

Memakai **robust z-score** berbasis median dan MAD (Median Absolute Deviation),
bukan mean/stdev, supaya beberapa hari ekstrem tidak menyembunyikan anomali hari lain:

```
robust_z = 0.6745 * (x - median) / MAD
```

Dihitung pada tiga dimensi: per tahap funnel, per kontraktor, dan per ROM.
Baris dengan `abs(robust_z) >= ANOMALY_Z_THRESHOLD` ditandai `is_anomaly = true`
dan menjadi bukti (evidence) untuk suggestion engine.

In [0]:
MAD_CONST = 0.6745


def robust_z(df: DataFrame, value_col: str, part_cols: List[str]) -> DataFrame:
    """Tambah kolom median, mad, robust_z pada partisi tertentu."""
    wpart = Window.partitionBy(*part_cols) if part_cols else Window.partitionBy(F.lit(1))
    med = F.expr(f"percentile_approx({value_col}, 0.5)")
    out = df.withColumn("_median", med.over(wpart))
    out = out.withColumn("_abs_dev", F.abs(F.col(value_col) - F.col("_median")))
    out = out.withColumn("_mad", F.expr("percentile_approx(_abs_dev, 0.5)").over(wpart))
    out = out.withColumn(
        "robust_z",
        F.round(F.when(F.col("_mad") > 0,
                       F.lit(MAD_CONST) * (F.col(value_col) - F.col("_median")) / F.col("_mad"))
                 .otherwise(F.lit(0.0)), 2))
    return out


# --- (a) anomali per tahap funnel ---
stage_loss = (
    spark.table(tbl("gold", "gold_stage_reconciliation"))
    .filter(F.col("gap_type") != "BUFFER_CHANGE")
    .filter(F.col("loss_pct").isNotNull())
    .select("production_date",
            F.lit("STAGE").alias("dimension"),
            F.col("pair_label").alias("dimension_value"),
            F.col("loss_pct").alias("metric_value"),
            F.col("delta_ton").alias("loss_ton"),
            F.col("upstream_ton").alias("reference_ton"))
)
stage_loss = robust_z(stage_loss, "metric_value", ["dimension_value"])

# --- (b) anomali per kontraktor ---
contractor_loss = (
    spark.table(tbl("gold", "gold_contractor_daily"))
    .filter(F.col("avg_loss_pct").isNotNull())
    .groupBy("production_date", "contractor_name")
    .agg(F.round(F.avg("avg_loss_pct"), 2).alias("metric_value"),
         F.round(F.sum("total_loss_ton"), 3).alias("loss_ton"),
         F.round(F.sum("netto_ton"), 3).alias("reference_ton"))
    .select("production_date",
            F.lit("CONTRACTOR").alias("dimension"),
            F.col("contractor_name").alias("dimension_value"),
            "metric_value", "loss_ton", "reference_ton")
)
contractor_loss = robust_z(contractor_loss, "metric_value", ["dimension_value"])

# --- (c) anomali per ROM ---
rom_loss = (
    spark.table(tbl("gold", "gold_rom_daily"))
    .filter(F.col("delta_urise_vs_wim_pct").isNotNull())
    .select("production_date",
            F.lit("ROM").alias("dimension"),
            F.col("rom_code").alias("dimension_value"),
            F.col("delta_urise_vs_wim_pct").alias("metric_value"),
            F.col("delta_urise_vs_wim_ton").alias("loss_ton"),
            F.col("hauled_ton").alias("reference_ton"))
)
rom_loss = robust_z(rom_loss, "metric_value", ["dimension_value"])

gold_anomaly = (
    stage_loss.unionByName(contractor_loss).unionByName(rom_loss)
    .withColumn("is_anomaly", F.abs(F.col("robust_z")) >= F.lit(ANOMALY_Z_THRESHOLD))
    .withColumn("severity",
                F.when(F.abs(F.col("robust_z")) >= ANOMALY_Z_THRESHOLD * 1.67, F.lit("CRITICAL"))
                 .when(F.abs(F.col("robust_z")) >= ANOMALY_Z_THRESHOLD, F.lit("HIGH"))
                 .when(F.abs(F.col("metric_value")) >= LOSS_PCT_CRITICAL, F.lit("MEDIUM"))
                 .otherwise(F.lit("LOW")))
    .withColumnRenamed("_median", "baseline_median_pct")
    .withColumnRenamed("_mad", "mad_pct")
    .drop("_abs_dev")
    .withColumn("_generated_at", F.current_timestamp())
)

write_table(gold_anomaly, "gold", "gold_loss_anomaly",
            partition_by=["production_date"],
            comment="Deteksi loss tidak wajar per stage/kontraktor/ROM memakai robust z-score (MAD).")

display(gold_anomaly.filter(F.col("is_anomaly"))
        .orderBy(F.desc(F.abs(F.col("robust_z")))))

### 4.9 `gold_shipment_outlook` - prediksi shortfall

Proyeksi sederhana dan mudah dijelaskan ke manajemen (bukan black-box):

```
run_rate_7d      = rata-rata tonase harian 7 hari terakhir
projected_month  = MTD aktual + run_rate_7d x sisa hari
shortfall_ton    = target bulan - projected_month   (positif = kurang)
required_rate    = (target - MTD) / sisa hari       (laju yang harus dicapai)
```

In [0]:
ts_daily_tbl = (
    spark.table(tbl("gold", "gold_production_timeseries"))
    .filter(F.col("grain") == "DAILY")
    .select(F.col("period_start").alias("production_date"), "actual_ton")
)

max_date = ts_daily_tbl.agg(F.max("production_date")).collect()[0][0]
print(f"Hari produksi terakhir: {max_date}")

w7d = Window.orderBy("production_date").rowsBetween(-6, 0)

outlook_base = (
    ts_daily_tbl
    .withColumn("month", F.date_format("production_date", "yyyy-MM"))
    .withColumn("run_rate_7d", F.round(F.avg("actual_ton").over(w7d), 3))
    .withColumn("mtd_actual_ton",
                F.round(F.sum("actual_ton").over(
                    Window.partitionBy("month").orderBy("production_date")
                          .rowsBetween(Window.unboundedPreceding, 0)), 3))
)

gold_outlook = (
    outlook_base
    .withColumn("days_in_month",
                F.dayofmonth(F.last_day(F.col("production_date"))))
    .withColumn("day_of_month", F.dayofmonth(F.col("production_date")))
    .withColumn("days_remaining", F.col("days_in_month") - F.col("day_of_month"))
    .withColumn("month_target_ton", monthly_target(F.col("month")))
    .withColumn("projected_month_ton",
                F.round(F.col("mtd_actual_ton")
                        + F.col("run_rate_7d") * F.col("days_remaining"), 3))
    .withColumn("shortfall_ton",
                F.round(F.col("month_target_ton") - F.col("projected_month_ton"), 3))
    .withColumn("shortfall_pct", safe_pct("shortfall_ton", "month_target_ton"))
    .withColumn("required_rate_ton_per_day",
                F.round(F.when(F.col("days_remaining") > 0,
                               (F.col("month_target_ton") - F.col("mtd_actual_ton"))
                               / F.col("days_remaining")), 3))
    .withColumn("rate_gap_ton_per_day",
                F.round(F.col("required_rate_ton_per_day") - F.col("run_rate_7d"), 3))
    .withColumn("outlook_status",
                F.when(F.col("shortfall_ton") <= 0, F.lit("ON_TRACK"))
                 .when(F.col("shortfall_pct") <= 5, F.lit("AT_RISK"))
                 .otherwise(F.lit("SHORTFALL")))
    .withColumn("is_latest", F.col("production_date") == F.lit(max_date))
    .withColumn("_generated_at", F.current_timestamp())
)

write_table(gold_outlook, "gold", "gold_shipment_outlook",
            partition_by=["month"],
            comment="Proyeksi pencapaian bulanan & prediksi shortfall shipment (run-rate 7 hari).")

display(gold_outlook.orderBy(F.desc("production_date")))

### 4.10 `gold_kpi_snapshot` - kartu KPI headline

In [0]:
funnel_tbl = spark.table(tbl("gold", "gold_funnel_stage_daily"))

kpi_funnel = (
    funnel_tbl.groupBy("production_date")
    .pivot("stage_code", [s[1] for s in FUNNEL_STAGES])
    .agg(F.first("tonnage_ton"))
)

kpi_ops = (
    wim.groupBy("production_date")
    .agg(F.count(F.lit(1)).alias("total_trips"),
         F.countDistinct("truck").alias("active_trucks"),
         F.countDistinct("contractor_code").alias("active_contractors"),
         F.round(F.avg("netto_ton"), 3).alias("avg_payload_ton"),
         F.round(F.avg("rom_to_port_min"), 1).alias("avg_cycle_min"),
         F.sum(F.col("is_payload_anomaly").cast("int")).alias("payload_anomaly_trips"))
)

kpi_trace = (
    spark.table(tbl("silver", "silver_trip_reconciliation"))
    .groupBy("production_date")
    .agg(F.round(F.avg(F.col("is_matched_cp").cast("int")) * 100, 1).alias("cp_trace_rate_pct"),
         F.round(F.avg(F.col("is_matched_rom").cast("int")) * 100, 1).alias("rom_trace_rate_pct"),
         F.round(F.sum("delta_wim_vs_cp_ton"), 3).alias("traced_loss_ton"))
)

kpi_stock = (
    spark.table(tbl("gold", "gold_stockpile_balance_hourly"))
    .groupBy("production_date")
    # max_by = nilai pada jam terakhir hari itu (bukan urutan baris sembarang)
    .agg(F.round(F.expr("max_by(balance_ton, hour_start_utc)"), 3).alias("closing_stock_ton"),
         F.round(F.expr("max_by(days_of_cover, hour_start_utc)"), 2).alias("days_of_cover"),
         F.expr("max_by(buffer_status, hour_start_utc)").alias("buffer_status"))
)

kpi_anom = (
    spark.table(tbl("gold", "gold_loss_anomaly"))
    .groupBy("production_date")
    .agg(F.sum(F.col("is_anomaly").cast("int")).alias("anomaly_count"),
         F.sum(F.when(F.col("severity") == "CRITICAL", 1).otherwise(0)).alias("critical_anomalies"))
)

kpi_outlook = (
    spark.table(tbl("gold", "gold_shipment_outlook"))
    .select("production_date", "run_rate_7d", "projected_month_ton",
            "month_target_ton", "shortfall_ton", "outlook_status")
)

gold_kpi = (
    kpi_funnel
    .join(kpi_ops, "production_date", "left")
    .join(kpi_trace, "production_date", "left")
    .join(kpi_stock, "production_date", "left")
    .join(kpi_anom, "production_date", "left")
    .join(kpi_outlook, "production_date", "left")
    .withColumnRenamed("ROM_PRODUCTION", "rom_production_ton")
    .withColumnRenamed("HAULING_WEIGHED", "hauling_ton")
    .withColumnRenamed("HAULING_DIRECT_CP", "direct_cp_ton")
    .withColumnRenamed("STOCKPILE_IN", "stockpile_in_ton")
    .withColumnRenamed("STOCKPILE_OUT", "stockpile_out_ton")
    .withColumnRenamed("CP_INTAKE", "cp_intake_ton")
    .withColumnRenamed("SHIPMENT", "shipment_ton")
    .withColumn("end_to_end_recovery_pct", safe_pct("cp_intake_ton", "hauling_ton"))
    .withColumn("end_to_end_loss_ton",
                F.round(F.col("hauling_ton") - F.col("cp_intake_ton"), 3))
    .withColumn("direct_share_pct", safe_pct("direct_cp_ton", "hauling_ton"))
    .withColumn("achievement_pct",
                F.round(F.col("hauling_ton")
                        / (F.col("month_target_ton")
                           / F.dayofmonth(F.last_day(F.col("production_date")))) * 100, 2))
    .withColumn("_generated_at", F.current_timestamp())
)

write_table(gold_kpi, "gold", "gold_kpi_snapshot",
            partition_by=["production_date"],
            comment="Kartu KPI headline harian untuk dashboard UC-07.")

display(gold_kpi.orderBy(F.desc("production_date")))

---
## 5. SUGGESTION ENGINE (Mosaic AI)

Dua lapis, sengaja dipisah supaya rekomendasi tetap terbit walau endpoint LLM mati
dan supaya setiap saran punya **bukti angka**, bukan karangan model:

1. **Rule engine (deterministik)** - membaca tabel gold dan menghasilkan *kandidat*
   rekomendasi lengkap dengan bukti (`evidence_json`) dan estimasi dampak
   (`expected_gain_ton`). Ini yang menjamin angka selalu benar dan bisa ditelusuri.
2. **Mosaic AI (`ai_query`)** - menerima kandidat + konteks harian, lalu memilih
   **top 3**, mengurutkan berdasar dampak x kemudahan eksekusi, dan menuliskannya
   dalam bahasa manajemen (aksi konkret + pemilik aksi).

Jika `ai_query` gagal (endpoint mati, quota habis, output bukan JSON), notebook
otomatis memakai teks rule engine dan menandai `generated_by = 'RULE_ENGINE'`.

### Katalog aturan

| Kode | Kategori | Pemicu | Estimasi dampak |
|---|---|---|---|
| `CONTRACTOR_BALANCING` | Balancing | `productivity_gap_pct <= -15%` | ritase kurang x payload rata-rata |
| `PAYLOAD_UNDERLOAD` | Balancing | payload rata-rata < fleet - 5% | selisih payload x jumlah ritase |
| `ABNORMAL_LOSS` | Loss | `abs(robust_z) >= 3` di gold_loss_anomaly | 50% dari loss ton (asumsi separuh bisa dipulihkan) |
| `STAGE_LOSS` | Loss | `loss_pct` tahap >= ambang WARNING | selisih ton tahap |
| `SHIPMENT_SHORTFALL` | Shipment | proyeksi bulan < target | shortfall ton |
| `STOCKPILE_BUFFER` | Stockpile | `days_of_cover < 1.5` | konsumsi CP x kekurangan hari |
| `CP_IMBALANCE` | Throughput | rasio intake CP tertinggi:terendah > 3x | selisih ke rata-rata |
| `TRACEABILITY_GAP` | Data quality | `cp_trace_rate_pct < 80` | tonase tak tertelusur |

### 5.1 Rule engine - pembentukan kandidat

In [0]:
from pyspark.sql.types import ArrayType

CANDIDATE_COLS = ["production_date", "candidate_code", "category", "entity",
                  "severity_score", "expected_gain_ton", "evidence_json",
                  "default_action"]


def mk_candidate(df, code_, category, entity_col, severity_col,
                 gain_col, evidence_struct, action_col):
    return (df.select(
        F.col("production_date"),
        F.lit(code_).alias("candidate_code"),
        F.lit(category).alias("category"),
        entity_col.alias("entity"),
        F.round(severity_col.cast("double"), 2).alias("severity_score"),
        F.round(gain_col.cast("double"), 1).alias("expected_gain_ton"),
        F.to_json(evidence_struct).alias("evidence_json"),
        action_col.alias("default_action"),
    ))


g_contractor = spark.table(tbl("gold", "gold_contractor_daily"))
g_anomaly = spark.table(tbl("gold", "gold_loss_anomaly"))
g_outlook = spark.table(tbl("gold", "gold_shipment_outlook"))
g_stock = spark.table(tbl("gold", "gold_stockpile_balance_hourly"))
g_cp = spark.table(tbl("gold", "gold_cp_intake_daily"))
g_stage = spark.table(tbl("gold", "gold_stage_reconciliation"))
g_kpi = spark.table(tbl("gold", "gold_kpi_snapshot"))

candidates = []

# ---- R1: balancing kontraktor (ritase per truk di bawah fleet) ----
c1 = (g_contractor
      .filter(F.col("balancing_flag") == "UNDER_UTILISED")
      .groupBy("production_date", "contractor_name")
      .agg(F.round(F.avg("productivity_gap_pct"), 2).alias("gap_pct"),
           F.round(F.avg("trips_per_truck"), 2).alias("trips_per_truck"),
           F.round(F.avg("fleet_avg_trips_per_truck"), 2).alias("fleet_trips_per_truck"),
           F.max("active_trucks").alias("active_trucks"),
           F.round(F.avg("avg_payload_ton"), 2).alias("avg_payload_ton"),
           F.round(F.sum("netto_ton"), 1).alias("netto_ton"))
      .withColumn("missed_trips",
                  F.greatest(F.col("fleet_trips_per_truck") - F.col("trips_per_truck"),
                             F.lit(0.0)) * F.col("active_trucks")))
candidates.append(mk_candidate(
    c1, "CONTRACTOR_BALANCING", "BALANCING",
    F.col("contractor_name"),
    F.abs(F.col("gap_pct")),
    F.col("missed_trips") * F.col("avg_payload_ton"),
    F.struct("contractor_name", "gap_pct", "trips_per_truck", "fleet_trips_per_truck",
             "active_trucks", "avg_payload_ton", "netto_ton",
             F.round(F.col("missed_trips"), 1).alias("missed_trips")),
    F.concat(F.lit("Naikkan ritase "), F.col("contractor_name"), F.lit(" ("),
             F.round(F.col("trips_per_truck"), 2).cast("string"),
             F.lit(" trip/truk vs fleet "),
             F.round(F.col("fleet_trips_per_truck"), 2).cast("string"),
             F.lit("): cek antrean muat ROM, waktu tunggu di CP, dan kesiapan unit."))))

# ---- R2: payload di bawah rata-rata fleet ----
c2 = (g_contractor
      .filter(F.col("deviation_vs_fleet_pct") <= -5.0)
      .groupBy("production_date", "contractor_name")
      .agg(F.round(F.avg("deviation_vs_fleet_pct"), 2).alias("dev_pct"),
           F.round(F.avg("avg_payload_ton"), 2).alias("avg_payload_ton"),
           F.round(F.avg("fleet_avg_payload_ton"), 2).alias("fleet_payload_ton"),
           F.sum("trips").alias("trips"),
           F.sum("payload_anomaly_trips").alias("payload_anomaly_trips")))
candidates.append(mk_candidate(
    c2, "PAYLOAD_UNDERLOAD", "BALANCING",
    F.col("contractor_name"),
    F.abs(F.col("dev_pct")),
    (F.col("fleet_payload_ton") - F.col("avg_payload_ton")) * F.col("trips"),
    F.struct("contractor_name", "dev_pct", "avg_payload_ton", "fleet_payload_ton",
             "trips", "payload_anomaly_trips"),
    F.concat(F.lit("Payload rata-rata "), F.col("contractor_name"), F.lit(" "),
             F.col("avg_payload_ton").cast("string"), F.lit(" ton vs fleet "),
             F.col("fleet_payload_ton").cast("string"),
             F.lit(" ton. Briefing operator alat muat & cek kalibrasi bucket."))))

# ---- R3: loss tidak wajar (anomali statistik) ----
c3 = g_anomaly.filter(F.col("is_anomaly"))
candidates.append(mk_candidate(
    c3, "ABNORMAL_LOSS", "LOSS",
    F.concat(F.col("dimension"), F.lit(": "), F.col("dimension_value")),
    F.abs(F.col("robust_z")) * F.lit(10.0),
    F.abs(F.col("loss_ton")) * F.lit(0.5),
    F.struct("dimension", "dimension_value", "metric_value", "loss_ton",
             "reference_ton", "robust_z", "baseline_median_pct", "severity"),
    F.concat(F.lit("Loss tidak wajar pada "), F.col("dimension_value"),
             F.lit(": "), F.col("metric_value").cast("string"),
             F.lit("% (baseline "), F.col("baseline_median_pct").cast("string"),
             F.lit("%, robust-z "), F.col("robust_z").cast("string"),
             F.lit("). Audit ritase & timbangan pada jalur tersebut."))))

# ---- R4: loss antar tahap melewati ambang ----
c4 = g_stage.filter(F.col("severity").isin("WARNING", "CRITICAL"))
candidates.append(mk_candidate(
    c4, "STAGE_LOSS", "LOSS",
    F.col("pair_label"),
    F.abs(F.col("loss_pct")) * F.lit(5.0),
    F.abs(F.col("delta_ton")),
    F.struct("pair_label", "upstream_stage", "downstream_stage", "upstream_ton",
             "downstream_ton", "delta_ton", "loss_pct", "recovery_pct", "severity"),
    F.concat(F.lit("Selisih tahap "), F.col("pair_label"), F.lit(" = "),
             F.col("delta_ton").cast("string"), F.lit(" ton ("),
             F.col("loss_pct").cast("string"),
             F.lit("%). Rekonsiliasi tiket muat vs tiket timbang pada tahap ini."))))

# ---- R5: proyeksi shortfall shipment ----
c5 = g_outlook.filter(F.col("outlook_status") != "ON_TRACK")
candidates.append(mk_candidate(
    c5, "SHIPMENT_SHORTFALL", "SHIPMENT",
    F.concat(F.lit("Target bulan "), F.col("month")),
    F.least(F.coalesce(F.col("shortfall_pct"), F.lit(0.0)) * F.lit(4.0), F.lit(100.0)),
    F.col("shortfall_ton"),
    F.struct("month", "mtd_actual_ton", "run_rate_7d", "days_remaining",
             "month_target_ton", "projected_month_ton", "shortfall_ton",
             "required_rate_ton_per_day", "rate_gap_ton_per_day", "outlook_status"),
    F.concat(F.lit("Proyeksi bulan "), F.col("month"), F.lit(" = "),
             F.col("projected_month_ton").cast("string"), F.lit(" ton vs target "),
             F.col("month_target_ton").cast("string"), F.lit(" ton. Perlu "),
             F.col("required_rate_ton_per_day").cast("string"),
             F.lit(" ton/hari (run-rate sekarang "),
             F.col("run_rate_7d").cast("string"), F.lit(" ton/hari))."))))

# ---- R6: buffer stockpile menipis ----
c6 = (g_stock.groupBy("production_date")
      .agg(F.round(F.min("days_of_cover"), 2).alias("min_days_of_cover"),
           F.round(F.expr("max_by(balance_ton, hour_start_utc)"), 1).alias("closing_stock_ton"),
           F.round(F.max("avg_daily_cp_ton"), 1).alias("avg_daily_cp_ton"),
           F.expr("max_by(buffer_status, hour_start_utc)").alias("buffer_status"),
           F.first("is_relative_balance").alias("is_relative_balance"))
      .filter(F.col("min_days_of_cover") < F.lit(STOCKPILE_MIN_COVER_DAYS)))
candidates.append(mk_candidate(
    c6, "STOCKPILE_BUFFER", "STOCKPILE",
    F.lit("NORTH-STOCKPILE"),
    (F.lit(STOCKPILE_MIN_COVER_DAYS) - F.col("min_days_of_cover")) * F.lit(30.0),
    (F.lit(STOCKPILE_MIN_COVER_DAYS) - F.col("min_days_of_cover")) * F.col("avg_daily_cp_ton"),
    F.struct("min_days_of_cover", "closing_stock_ton", "avg_daily_cp_ton",
             "buffer_status", "is_relative_balance"),
    F.concat(F.lit("Buffer North Stockpile "), F.col("min_days_of_cover").cast("string"),
             F.lit(" hari (< "), F.lit(str(STOCKPILE_MIN_COVER_DAYS)),
             F.lit(" hari). Tambah alokasi hauling ke stockpile atau tahan reclaim."))))

# ---- R7: ketimpangan beban antar Crushing Plant ----
cp_day = (g_cp.groupBy("production_date", "cp_code")
          .agg(F.round(F.sum("intake_ton"), 1).alias("intake_ton")))
wcpd = Window.partitionBy("production_date")
c7 = (cp_day
      .withColumn("max_ton", F.max("intake_ton").over(wcpd))
      .withColumn("min_ton", F.min("intake_ton").over(wcpd))
      .withColumn("avg_ton", F.round(F.avg("intake_ton").over(wcpd), 1))
      .withColumn("imbalance_ratio",
                  F.round(F.col("max_ton") / F.greatest(F.col("min_ton"), F.lit(1.0)), 2))
      .filter((F.col("imbalance_ratio") > 3.0) & (F.col("intake_ton") == F.col("min_ton"))))
candidates.append(mk_candidate(
    c7, "CP_IMBALANCE", "THROUGHPUT",
    F.col("cp_code"),
    F.least(F.col("imbalance_ratio") * F.lit(8.0), F.lit(100.0)),
    F.col("avg_ton") - F.col("intake_ton"),
    F.struct("cp_code", "intake_ton", "min_ton", "max_ton", "avg_ton", "imbalance_ratio"),
    F.concat(F.lit("Intake "), F.col("cp_code"), F.lit(" hanya "),
             F.col("intake_ton").cast("string"), F.lit(" ton vs rata-rata "),
             F.col("avg_ton").cast("string"),
             F.lit(" ton. Alihkan sebagian ritase ke CP ini bila hopper & conveyor siap."))))

# ---- R8: keterlacakan trip rendah (data quality) ----
c8 = (g_kpi.filter(F.col("cp_trace_rate_pct") < 80.0)
      .select("production_date", "cp_trace_rate_pct", "rom_trace_rate_pct",
              "hauling_ton", "cp_intake_ton", "end_to_end_loss_ton"))
candidates.append(mk_candidate(
    c8, "TRACEABILITY_GAP", "DATA_QUALITY",
    F.lit("Trip tracing WIM -> CP"),
    (F.lit(100.0) - F.col("cp_trace_rate_pct")),
    F.col("end_to_end_loss_ton"),
    F.struct("cp_trace_rate_pct", "rom_trace_rate_pct", "hauling_ton",
             "cp_intake_ton", "end_to_end_loss_ton"),
    F.concat(F.lit("Hanya "), F.col("cp_trace_rate_pct").cast("string"),
             F.lit("% trip WIM yang ketemu event dumping CP. Cek reader RFID CP "
                   "dan sinkronisasi jam perangkat sebelum angka loss dipakai."))))

df_candidates = candidates[0]
for c in candidates[1:]:
    df_candidates = df_candidates.unionByName(c)

df_candidates = (
    df_candidates
    .filter(F.col("production_date").isNotNull())
    .withColumn("expected_gain_ton",
                F.coalesce(F.col("expected_gain_ton"), F.lit(0.0)))
    .withColumn("severity_score",
                F.coalesce(F.col("severity_score"), F.lit(0.0)))
    # skor gabungan: separuh urgensi, separuh dampak tonase (dinormalisasi harian)
    .withColumn("_max_gain", F.max("expected_gain_ton").over(Window.partitionBy("production_date")))
    .withColumn("impact_score",
                F.round(F.when(F.col("_max_gain") > 0,
                               F.col("expected_gain_ton") / F.col("_max_gain") * 100.0)
                         .otherwise(F.lit(0.0)), 2))
    .withColumn("priority_score",
                F.round(F.col("severity_score") * 0.5 + F.col("impact_score") * 0.5, 2))
    .drop("_max_gain")
    .withColumn("candidate_rank",
                F.row_number().over(Window.partitionBy("production_date")
                                    .orderBy(F.desc("priority_score"),
                                             F.desc("expected_gain_ton"))))
    .withColumn("_generated_at", F.current_timestamp())
)

write_table(df_candidates, "gold", "gold_suggestion_candidates",
            partition_by=["production_date"],
            comment="Kandidat rekomendasi deterministik (rule engine) + bukti angka per hari.")

display(df_candidates.orderBy(F.desc("production_date"), "candidate_rank"))

### 5.2 Payload harian untuk Mosaic AI

Konteks yang dikirim ke model dibatasi: KPI hari itu + maksimal 8 kandidat teratas.
Prompt memaksa model **hanya memilih dan menarasikan dari kandidat yang diberikan**
(tidak boleh mengarang angka), dan output wajib JSON array.

In [0]:
MAX_CANDIDATES_TO_LLM = 8

cand_top = (spark.table(tbl("gold", "gold_suggestion_candidates"))
            .filter(F.col("candidate_rank") <= MAX_CANDIDATES_TO_LLM))

cand_payload = (
    cand_top.groupBy("production_date")
    .agg(F.to_json(F.collect_list(F.struct(
        F.col("candidate_rank").alias("rank"),
        F.col("candidate_code").alias("code"),
        F.col("category"),
        F.col("entity"),
        F.col("expected_gain_ton"),
        F.col("priority_score"),
        F.col("default_action"),
        F.col("evidence_json").alias("evidence"),
    ))).alias("candidates_json"))
)

kpi_payload = (
    spark.table(tbl("gold", "gold_kpi_snapshot"))
    .select("production_date",
            F.to_json(F.struct(
                "rom_production_ton", "hauling_ton", "direct_cp_ton",
                "stockpile_in_ton", "stockpile_out_ton", "cp_intake_ton",
                "shipment_ton", "end_to_end_loss_ton", "end_to_end_recovery_pct",
                "total_trips", "active_trucks", "active_contractors",
                "avg_payload_ton", "avg_cycle_min", "closing_stock_ton",
                "days_of_cover", "buffer_status", "run_rate_7d",
                "projected_month_ton", "month_target_ton", "shortfall_ton",
                "outlook_status", "cp_trace_rate_pct", "anomaly_count",
            )).alias("kpi_json"))
)

SYSTEM_RULES = (
    "Anda analis operasional tambang batu bara di PT Borneo Indobara. "
    "Tugas: memilih 3 rekomendasi harian paling berdampak untuk meningkatkan throughput "
    "produksi end-to-end (ROM -> hauling -> stockpile -> crushing plant -> shipment).\
"
    "ATURAN KETAT:\
"
    "1. Pilih HANYA dari daftar KANDIDAT yang diberikan. Dilarang membuat rekomendasi baru.\
"
    "2. Dilarang mengarang angka. Semua angka harus berasal dari KPI atau evidence kandidat.\
"
    "3. Urutkan berdasarkan dampak tonase x kemudahan eksekusi dalam 24 jam.\
"
    "4. Setiap rekomendasi harus menyebut aksi konkret dan pemilik aksi "
    "(Manajer Produksi / Manajer Crushing Plant / Tim Perencanaan / Tim Shipping / "
    "Pengawas Hauling / Tim Data).\
"
    "5. Bahasa Indonesia, ringkas, maksimal 2 kalimat per field.\
"
    "6. Jawab HANYA JSON array valid, tanpa markdown, tanpa penjelasan tambahan.\
"
    'Format: [{"rank":1,"candidate_code":"...","category":"...",'
    '"title":"...","recommendation":"...","rationale":"...",'
    '"owner":"...","expected_gain_ton":0.0,"priority":"HIGH|MEDIUM|LOW",'
    '"confidence":0.0}]'
)

llm_input = (
    kpi_payload.join(cand_payload, "production_date", "inner")
    .withColumn("prompt",
                F.concat(
                    F.lit(SYSTEM_RULES),
                    F.lit("\
\
=== TANGGAL PRODUKSI ==="), F.lit("\
"),
                    F.col("production_date").cast("string"),
                    F.lit("\
\
=== KPI HARI INI (ton) ===\
"), F.col("kpi_json"),
                    F.lit("\
\
=== KANDIDAT REKOMENDASI ===\
"), F.col("candidates_json"),
                    F.lit(f"\
\
Keluarkan tepat {SUGGESTION_TOP_N} rekomendasi terbaik "
                          "sebagai JSON array.")))
)

print(f"Hari yang akan diproses suggestion engine: {llm_input.count()}")
display(llm_input.select("production_date", F.length("prompt").alias("prompt_chars"))
        .orderBy(F.desc("production_date")))

### 5.3 Panggilan Mosaic AI (`ai_query`) + fallback

`ai_query(endpoint, prompt)` dijalankan terdistribusi di Spark. Pada DBR yang
mendukung parameter bernama, versi yang lebih tahan gagal adalah:

```sql
ai_query('<endpoint>', prompt, failOnError => false,
         modelParameters => named_struct('temperature', 0.1, 'max_tokens', 1200))
```

Notebook mencoba bentuk lengkap dulu, lalu turun ke bentuk 2 argumen, lalu ke
rule engine murni.

In [0]:
SUGGESTION_SCHEMA = ArrayType(StructType([
    StructField("rank", IntegerType()),
    StructField("candidate_code", StringType()),
    StructField("category", StringType()),
    StructField("title", StringType()),
    StructField("recommendation", StringType()),
    StructField("rationale", StringType()),
    StructField("owner", StringType()),
    StructField("expected_gain_ton", DoubleType()),
    StructField("priority", StringType()),
    StructField("confidence", DoubleType()),
]))


def call_mosaic_ai(df: DataFrame) -> Tuple[Optional[DataFrame], str]:
    """Kembalikan (df_dengan_kolom_llm_response, mode) atau (None, alasan gagal)."""
    if not SUGGESTION_ENABLED:
        return None, "SUGGESTION_ENABLED=False"

    attempts = [
        ("ai_query_full",
         f"ai_query('{MOSAIC_AI_ENDPOINT}', prompt, failOnError => false, "
         f"modelParameters => named_struct('temperature', 0.1, 'max_tokens', 1200))"),
        ("ai_query_simple",
         f"ai_query('{MOSAIC_AI_ENDPOINT}', prompt)"),
    ]
    for mode, expr in attempts:
        try:
            out = df.withColumn("llm_response", F.expr(expr).cast("string"))
            out = out.cache()
            out.count()          # paksa eksekusi supaya error muncul di sini
            print(f"Mosaic AI OK via {mode} (endpoint: {MOSAIC_AI_ENDPOINT})")
            return out, mode
        except Exception as e:                       # noqa: BLE001
            print(f"[WARN] {mode} gagal: {type(e).__name__}: {str(e)[:180]}")
    return None, "ai_query gagal pada semua percobaan"


llm_out, llm_mode = call_mosaic_ai(llm_input)

if llm_out is not None:
    parsed = (
        llm_out
        # ambil JSON array pertama dari respons (buang pagar markdown bila ada)
        .withColumn("_json",
                    F.regexp_extract(F.col("llm_response"), r"(?s)\[\s*\{.*\}\s*\]", 0))
        .withColumn("suggestions", F.from_json(F.col("_json"), SUGGESTION_SCHEMA))
        .withColumn("parse_ok",
                    F.col("suggestions").isNotNull() & (F.size("suggestions") >= 1))
    )
    ok_days = parsed.filter(F.col("parse_ok")).count()
    total_days = parsed.count()
    print(f"Respons LLM berhasil di-parse: {ok_days}/{total_days} hari")
else:
    parsed = None
    print(f"Suggestion engine memakai RULE_ENGINE saja ({llm_mode}).")

### 5.4 `gold_suggestion_daily` - top 3 rekomendasi harian

Satu baris = satu rekomendasi. `generated_by` membedakan hasil Mosaic AI dan
fallback rule engine, `evidence_json` menyimpan angka pendukung sehingga setiap
saran bisa diaudit sampai ke tabel silver.

In [0]:
cand_ref = spark.table(tbl("gold", "gold_suggestion_candidates"))

# ---------- Jalur A: hasil Mosaic AI ----------
sugg_ai = None
if parsed is not None:
    sugg_ai = (
        parsed.filter(F.col("parse_ok"))
        .select("production_date", F.explode("suggestions").alias("s"))
        .select(
            "production_date",
            F.col("s.rank").alias("rank"),
            F.col("s.candidate_code").alias("candidate_code"),
            F.col("s.category").alias("category"),
            F.col("s.title").alias("title"),
            F.col("s.recommendation").alias("recommendation"),
            F.col("s.rationale").alias("rationale"),
            F.coalesce(F.col("s.owner"), F.lit("Manajer Produksi")).alias("owner"),
            F.round(F.coalesce(F.col("s.expected_gain_ton"), F.lit(0.0)), 1)
             .alias("expected_gain_ton"),
            F.coalesce(F.col("s.priority"), F.lit("MEDIUM")).alias("priority"),
            F.round(F.coalesce(F.col("s.confidence"), F.lit(0.7)), 2).alias("confidence"),
            F.lit("MOSAIC_AI").alias("generated_by"),
        )
        .filter(F.col("rank") <= SUGGESTION_TOP_N)
    )

# ---------- Jalur B: fallback rule engine ----------
OWNER_BY_CATEGORY = {
    "BALANCING":    "Pengawas Hauling / Manajer Produksi",
    "LOSS":         "Manajer Produksi / Manajer Quality",
    "SHIPMENT":     "Tim Perencanaan / Tim Shipping",
    "STOCKPILE":    "Manajer Crushing Plant",
    "THROUGHPUT":   "Manajer Crushing Plant",
    "DATA_QUALITY": "Tim Data / IT Operasional",
}
OWNER_EXPR = F.create_map([x for kv in OWNER_BY_CATEGORY.items()
                           for x in (F.lit(kv[0]), F.lit(kv[1]))])

sugg_rule = (
    cand_ref.filter(F.col("candidate_rank") <= SUGGESTION_TOP_N)
    .select(
        "production_date",
        F.col("candidate_rank").alias("rank"),
        "candidate_code", "category",
        F.concat(F.col("candidate_code"), F.lit(" - "), F.col("entity")).alias("title"),
        F.col("default_action").alias("recommendation"),
        F.concat(F.lit("Skor prioritas "), F.col("priority_score").cast("string"),
                 F.lit(" (urgensi "), F.col("severity_score").cast("string"),
                 F.lit(", dampak "), F.col("impact_score").cast("string"),
                 F.lit("). Potensi "), F.col("expected_gain_ton").cast("string"),
                 F.lit(" ton.")).alias("rationale"),
        F.coalesce(OWNER_EXPR[F.col("category")], F.lit("Manajer Produksi")).alias("owner"),
        "expected_gain_ton",
        F.when(F.col("priority_score") >= 66, F.lit("HIGH"))
         .when(F.col("priority_score") >= 33, F.lit("MEDIUM"))
         .otherwise(F.lit("LOW")).alias("priority"),
        F.lit(1.0).alias("confidence"),
        F.lit("RULE_ENGINE").alias("generated_by"),
    )
)

if sugg_ai is not None:
    ai_days = [r[0] for r in sugg_ai.select("production_date").distinct().collect()]
    sugg_all = sugg_ai.unionByName(
        sugg_rule.filter(~F.col("production_date").isin(ai_days))
        if ai_days else sugg_rule)
else:
    sugg_all = sugg_rule

# Lampirkan kembali bukti angka dari kandidat (audit trail).
# Satu kode kandidat bisa muncul untuk beberapa entity dalam satu hari
# (mis. balancing untuk 3 kontraktor) -> ambil yang peringkatnya paling tinggi
# supaya join tidak menggandakan baris rekomendasi.
evidence = (
    cand_ref
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("production_date", "candidate_code")
              .orderBy("candidate_rank")))
    .filter(F.col("_rn") == 1)
    .select(
        "production_date",
        F.col("candidate_code"),
        F.col("entity"),
        F.col("evidence_json"),
        F.col("priority_score"),
        F.col("expected_gain_ton").alias("rule_expected_gain_ton"),
    )
)

gold_suggestion = (
    sugg_all.join(evidence, ["production_date", "candidate_code"], "left")
    .withColumn("expected_gain_ton",
                F.round(F.coalesce(F.col("rule_expected_gain_ton"),
                                   F.col("expected_gain_ton")), 1))
    .withColumn("model_endpoint",
                F.when(F.col("generated_by") == "MOSAIC_AI", F.lit(MOSAIC_AI_ENDPOINT))
                 .otherwise(F.lit(None).cast("string")))
    .withColumn("engine_mode", F.lit(llm_mode))
    .withColumn("_generated_at", F.current_timestamp())
    .dropDuplicates(["production_date", "rank"])
    .select("production_date", "rank", "candidate_code", "category", "entity",
            "title", "recommendation", "rationale", "owner",
            "expected_gain_ton", "priority", "confidence",
            "evidence_json", "priority_score",
            "generated_by", "model_endpoint", "engine_mode", "_generated_at")
)

write_table(gold_suggestion, "gold", "gold_suggestion_daily",
            partition_by=["production_date"],
            comment="Top-3 rekomendasi harian peningkatan throughput (Mosaic AI + rule engine).")

display(gold_suggestion.orderBy(F.desc("production_date"), "rank"))

In [0]:
# Tampilan siap-tempel untuk morning briefing
latest_day = (spark.table(tbl("gold", "gold_suggestion_daily"))
              .agg(F.max("production_date")).collect()[0][0])

rows = (spark.table(tbl("gold", "gold_suggestion_daily"))
        .filter(F.col("production_date") == F.lit(latest_day))
        .orderBy("rank").collect())

print("=" * 78)
print(f"TOP {SUGGESTION_TOP_N} REKOMENDASI - hari produksi {latest_day}")
print("=" * 78)
for r in rows:
    print(f"\
[{r['rank']}] {r['priority']:<6} | {r['category']:<12} | "
          f"potensi {r['expected_gain_ton']:,.1f} ton")
    print(f"    {r['title']}")
    print(f"    Aksi   : {r['recommendation']}")
    print(f"    Alasan : {r['rationale']}")
    print(f"    Pemilik: {r['owner']}   (sumber: {r['generated_by']})")
print("\
" + "=" * 78)

---
## 6. Data quality & audit

Kualitas data ditulis ke tabel supaya bisa dipantau dari dashboard - bukan sekadar
`assert` yang mati di notebook. Aturan: cek yang **BLOCKING** menghentikan run
(angka funnel tidak boleh terbit kalau salah), cek **WARNING** hanya dicatat.

In [0]:
dq_rows = []


def dq(check: str, layer: str, table: str, level: str, value, threshold, passed: bool,
       detail: str = ""):
    dq_rows.append((check, layer, table, level, float(value) if value is not None else None,
                    float(threshold) if threshold is not None else None, bool(passed), detail))


sw = spark.table(tbl("silver", "silver_wim_hauling"))
su = spark.table(tbl("silver", "silver_urise_events"))
ss = spark.table(tbl("silver", "silver_uscavis_stock_txn"))
sr = spark.table(tbl("silver", "silver_trip_reconciliation"))
gf = spark.table(tbl("gold", "gold_funnel_stage_daily"))

# 1. Tidak ada trip_id ganda (BLOCKING)
dup_trips = sw.count() - sw.select("trip_id").distinct().count()
dq("unique_trip_id", "silver", "silver_wim_hauling", "BLOCKING",
   dup_trips, 0, dup_trips == 0, "trip_id duplikat setelah dedup")

# 2. Tonase WIM tidak boleh nol/negatif (BLOCKING)
bad_netto = sw.filter((F.col("netto_ton").isNull()) | (F.col("netto_ton") <= 0)).count()
dq("netto_positive", "silver", "silver_wim_hauling", "BLOCKING",
   bad_netto, 0, bad_netto == 0, "baris netto null/<=0")

# 3. Payload di luar rentang wajar (WARNING)
anom_payload = sw.filter(F.col("is_payload_anomaly")).count()
pct_anom = anom_payload / max(sw.count(), 1) * 100
dq("payload_in_range", "silver", "silver_wim_hauling", "WARNING",
   round(pct_anom, 2), 2.0, pct_anom <= 2.0,
   f"{anom_payload} trip di luar {PAYLOAD_MIN_TON}-{PAYLOAD_MAX_TON} ton")

# 4. ROM tidak dikenali (WARNING)
unknown_rom = sw.filter(F.col("is_rom_unknown")).count()
pct_rom = unknown_rom / max(sw.count(), 1) * 100
dq("rom_resolved", "silver", "silver_wim_hauling", "WARNING",
   round(pct_rom, 2), 5.0, pct_rom <= 5.0, f"{unknown_rom} trip tanpa ROM valid")

# 5. Event Urise yang node-nya tidak terpetakan (WARNING)
unresolved = su.filter(F.col("node_type") == "UNRESOLVED").count()
pct_unres = unresolved / max(su.count(), 1) * 100
dq("urise_node_resolved", "silver", "silver_urise_events", "WARNING",
   round(pct_unres, 2), 5.0, pct_unres <= 5.0,
   f"{unresolved} event tanpa master_location")

# 6. Mutasi stok bernilai nol (WARNING)
zero_qty = ss.filter(F.col("is_zero_qty")).count()
pct_zero = zero_qty / max(ss.count(), 1) * 100
dq("stock_qty_positive", "silver", "silver_uscavis_stock_txn", "WARNING",
   round(pct_zero, 2), 10.0, pct_zero <= 10.0, f"{zero_qty} mutasi qty <= 0")

# 7. Keterlacakan trip ke CP (WARNING) - dipakai untuk menilai kepercayaan angka loss
trace_rate = sr.agg(F.avg(F.col("is_matched_cp").cast("int")) * 100).collect()[0][0] or 0.0
dq("trip_trace_rate", "silver", "silver_trip_reconciliation", "WARNING",
   round(trace_rate, 2), 80.0, trace_rate >= 80.0, "% trip WIM yang ketemu event CP")

# 8. Funnel: CP intake tidak boleh melebihi hauling + reclaim (BLOCKING)
piv = (gf.groupBy("production_date").pivot("stage_code",
       ["HAULING_WEIGHED", "STOCKPILE_OUT", "CP_INTAKE"])
       .agg(F.first("tonnage_ton")))
overflow_days = piv.filter(
    F.col("CP_INTAKE") > (F.coalesce(F.col("HAULING_WEIGHED"), F.lit(0.0))
                          + F.coalesce(F.col("STOCKPILE_OUT"), F.lit(0.0))) * 1.05
).count()
dq("funnel_monotonic", "gold", "gold_funnel_stage_daily", "BLOCKING",
   overflow_days, 0, overflow_days == 0,
   "hari dengan CP intake > (hauling + reclaim) + 5% toleransi")

# 9. Tidak ada hari bolong di time-series (WARNING)
ts = spark.table(tbl("gold", "gold_production_timeseries")).filter(F.col("grain") == "DAILY")
d_min, d_max, d_cnt = ts.agg(F.min("period_start"), F.max("period_start"),
                             F.countDistinct("period_start")).collect()[0]
expected_days = (d_max - d_min).days + 1 if d_min and d_max else 0
dq("no_missing_days", "gold", "gold_production_timeseries", "WARNING",
   expected_days - d_cnt, 0, (expected_days - d_cnt) == 0,
   f"{d_min} .. {d_max}: {d_cnt}/{expected_days} hari terisi")

dq_schema = StructType([
    StructField("check_name", StringType()),
    StructField("layer", StringType()),
    StructField("table_name", StringType()),
    StructField("level", StringType()),
    StructField("value", DoubleType()),
    StructField("threshold", DoubleType()),
    StructField("passed", BooleanType()),
    StructField("detail", StringType()),
])

df_dq = (spark.createDataFrame(dq_rows, dq_schema)
         .withColumn("run_ts_utc", F.lit(RUN_TS).cast("timestamp"))
         .withColumn("run_date", F.current_date()))

write_table(df_dq, "gold", "gold_data_quality_log", mode="append",
            comment="Log hasil pemeriksaan kualitas data tiap run UC-07.")

display(df_dq)

blocking_failed = [r for r in dq_rows if r[3] == "BLOCKING" and not r[6]]
warning_failed = [r for r in dq_rows if r[3] == "WARNING" and not r[6]]
print(f"\
BLOCKING gagal : {len(blocking_failed)}")
print(f"WARNING  gagal : {len(warning_failed)}")
for r in warning_failed:
    print(f"  [WARN] {r[0]}: value={r[4]} threshold={r[5]} - {r[7]}")
if blocking_failed:
    for r in blocking_failed:
        print(f"  [FAIL] {r[0]}: value={r[4]} threshold={r[5]} - {r[7]}")
    raise ValueError(f"{len(blocking_failed)} pemeriksaan BLOCKING gagal - "
                     f"tabel gold jangan dipublikasikan sebelum diperbaiki.")
print("Semua pemeriksaan BLOCKING lulus.")

---
## 7. Retensi 3 bulan + arsip PostgreSQL

Spesifikasi use case: **data window 3 bulan di Delta**, periode lebih panjang
diarsipkan ke PostgreSQL. Urutan yang aman dan tidak menghilangkan data:

1. Salin partisi yang lebih tua dari 3 bulan ke PostgreSQL (`append`).
2. Verifikasi jumlah baris di PostgreSQL sama dengan sumber.
3. Baru hapus partisi lama dari Delta.

Langkah 3 tidak akan berjalan bila langkah 1-2 gagal atau
`PG_ARCHIVE_ENABLED = False`.

In [0]:
ARCHIVE_TABLES = [
    ("gold", "gold_funnel_stage_daily",     "production_date"),
    ("gold", "gold_stage_reconciliation",   "production_date"),
    ("gold", "gold_contractor_daily",       "production_date"),
    ("gold", "gold_rom_daily",              "production_date"),
    ("gold", "gold_cp_intake_daily",        "production_date"),
    ("gold", "gold_stockpile_balance_hourly", "production_date"),
    ("gold", "gold_kpi_snapshot",           "production_date"),
    ("gold", "gold_suggestion_daily",       "production_date"),
    ("silver", "silver_wim_hauling",        "production_date"),
    ("silver", "silver_trip_reconciliation", "production_date"),
]

cutoff_date = (spark.sql(
    f"SELECT add_months(current_date(), -{DELTA_RETENTION_MONTHS}) AS c")
    .collect()[0]["c"])
print(f"Cutoff retensi Delta: {cutoff_date} "
      f"(partisi lebih tua diarsipkan lalu dihapus)")


def pg_props() -> Dict[str, str]:
    return {
        "user": dbutils.secrets.get(PG_SECRET_SCOPE, PG_USER_KEY),
        "password": dbutils.secrets.get(PG_SECRET_SCOPE, PG_PASS_KEY),
        "driver": "org.postgresql.Driver",
    }


def archive_and_prune(layer: str, name: str, date_col: str) -> Dict:
    fqn = tbl(layer, name)
    old = spark.table(fqn).filter(F.col(date_col) < F.lit(cutoff_date))
    n_old = old.count()
    result = {"table": fqn, "rows_older_than_cutoff": n_old,
              "archived": False, "pruned": False}
    if n_old == 0:
        return result
    if not PG_ARCHIVE_ENABLED:
        print(f"  {fqn}: {n_old:,} baris tua, arsip PG nonaktif -> tidak dihapus")
        return result

    pg_table = f"{PG_SCHEMA}.{name}"
    (old.write.mode("append").jdbc(PG_JDBC_URL, pg_table, properties=pg_props()))

    n_pg = (spark.read.jdbc(
        PG_JDBC_URL,
        f"(SELECT count(*) AS n FROM {pg_table} WHERE {date_col} < DATE '{cutoff_date}') t",
        properties=pg_props()).collect()[0]["n"])
    result["archived"] = True
    result["rows_in_pg"] = int(n_pg)

    if int(n_pg) >= n_old:
        spark.sql(f"DELETE FROM {fqn} WHERE {date_col} < DATE '{cutoff_date}'")
        result["pruned"] = True
        print(f"  {fqn}: {n_old:,} baris diarsipkan ke {pg_table} lalu dihapus dari Delta")
    else:
        print(f"  [WARN] {fqn}: verifikasi gagal (PG {n_pg:,} < sumber {n_old:,}); "
              f"partisi Delta TIDAK dihapus")
    return result


archive_results = []
for layer, name, dcol in ARCHIVE_TABLES:
    try:
        archive_results.append(archive_and_prune(layer, name, dcol))
    except Exception as e:                              # noqa: BLE001
        print(f"  [ERROR] {name}: {type(e).__name__}: {str(e)[:160]}")
        archive_results.append({"table": name, "error": str(e)[:160]})

for r in archive_results:
    print(r)

In [0]:
# OPTIMIZE + VACUUM tabel gold (jalankan sekali per hari, bukan tiap jam)
RUN_MAINTENANCE = False

if RUN_MAINTENANCE:
    for layer, name, _dcol in ARCHIVE_TABLES:
        fqn = tbl(layer, name)
        spark.sql(f"OPTIMIZE {fqn}")
        spark.sql(f"VACUUM {fqn} RETAIN 168 HOURS")
        print(f"  maintained: {fqn}")
else:
    print("RUN_MAINTENANCE=False - lewati OPTIMIZE/VACUUM "
          "(jadwalkan di job harian terpisah).")

---
## 8. Referensi query dashboard

Query siap pakai untuk Databricks SQL / Lakeview. Semua sudah membaca gold,
tidak ada join berat di sisi dashboard.

In [0]:
# Query di bawah dicetak untuk disalin ke Databricks SQL / Lakeview.
# Parameter :date_from, :date_to, :grain, :production_date diisi lewat widget dashboard.

FUNNEL_SQL = f"""
-- Visual 1: Funnel ROM -> Hauling -> Stockpile -> CP -> Shipment (rentang tanggal)
SELECT stage_seq, stage_code, stage_name, stage_branch, source_system,
       ROUND(SUM(tonnage_ton), 1) AS tonnage_ton,
       SUM(trips)                 AS trips,
       ROUND(AVG(pct_of_baseline), 1) AS pct_of_hauling
FROM {tbl('gold', 'gold_funnel_stage_daily')}
WHERE production_date BETWEEN :date_from AND :date_to
GROUP BY stage_seq, stage_code, stage_name, stage_branch, source_system
ORDER BY stage_seq
"""

LOSS_SQL = f"""
-- Visual 2: Selisih volumetrik & % loss/recovery per tahap
SELECT production_date, pair_label, gap_type,
       upstream_ton, downstream_ton, delta_ton, loss_pct, recovery_pct, severity
FROM {tbl('gold', 'gold_stage_reconciliation')}
WHERE production_date BETWEEN :date_from AND :date_to
ORDER BY production_date DESC, ABS(loss_pct) DESC
"""

TREND_SQL = f"""
-- Visual 3: Produksi mingguan/bulanan vs target
SELECT grain, period_label, period_start, actual_ton, target_ton,
       variance_ton, achievement_pct, ma7_actual_ton, status
FROM {tbl('gold', 'gold_production_timeseries')}
WHERE grain = :grain
ORDER BY period_start
"""

CONTRACTOR_SQL = f"""
-- Visual 4: Balancing kontraktor
SELECT production_date, contractor_name, netto_ton, trips, active_trucks,
       trips_per_truck, fleet_avg_trips_per_truck, productivity_gap_pct,
       avg_payload_ton, share_pct, avg_loss_pct, balancing_flag
FROM {tbl('gold', 'gold_contractor_daily')}
WHERE production_date = :production_date
ORDER BY netto_ton DESC
"""

STOCK_SQL = f"""
-- Visual 5: Saldo North Stockpile & days-of-cover
SELECT hour_start_utc, in_ton, out_ton, net_ton, balance_ton,
       days_of_cover, buffer_status
FROM {tbl('gold', 'gold_stockpile_balance_hourly')}
WHERE production_date BETWEEN :date_from AND :date_to
ORDER BY hour_start_utc
"""

SUGGESTION_SQL = f"""
-- Visual 6: Top 3 rekomendasi harian
SELECT rank, priority, category, title, recommendation, rationale,
       owner, expected_gain_ton, confidence, generated_by
FROM {tbl('gold', 'gold_suggestion_daily')}
WHERE production_date = (SELECT MAX(production_date)
                         FROM {tbl('gold', 'gold_suggestion_daily')})
ORDER BY rank
"""

for label, q in [("FUNNEL", FUNNEL_SQL), ("LOSS", LOSS_SQL), ("TREND", TREND_SQL),
                 ("CONTRACTOR", CONTRACTOR_SQL), ("STOCK", STOCK_SQL),
                 ("SUGGESTION", SUGGESTION_SQL)]:
    print(f"\
----- {label} -----{q}")

In [0]:
# Contoh eksekusi funnel (tanpa parameter :date_from/:date_to)
display(spark.sql(f"""
    SELECT stage_seq, stage_code, stage_name, source_system,
           ROUND(SUM(tonnage_ton), 1) AS tonnage_ton,
           SUM(trips) AS trips
    FROM {tbl('gold', 'gold_funnel_stage_daily')}
    GROUP BY stage_seq, stage_code, stage_name, source_system
    ORDER BY stage_seq
"""))

---
## 9. Orkestrasi (refresh per jam + suggestion harian)

Karakteristik use case: **batch per jam** untuk visualisasi, **harian** untuk
suggestion engine. Pemisahan job disarankan supaya panggilan LLM tidak ikut
16x sehari.

```
Job: uc07_hourly_refresh          (cron: 0 5 * * * *  -> menit ke-5 tiap jam)
  task_1 ingest_bronze            -> section 2
  task_2 build_silver             -> section 3   (depends: task_1)
  task_3 build_gold               -> section 4   (depends: task_2)
  task_4 data_quality             -> section 6   (depends: task_3)

Job: uc07_daily_suggestion        (cron: 0 30 6 * * ?  -> 06:30 WITA, awal shift pagi)
  task_1 suggestion_engine        -> section 5
  task_2 archive_and_retention    -> section 7   (jadwalkan mingguan saja bila perlu)
```

Untuk menjalankan hanya sebagian section, pakai widget `run_mode` di sel berikut.

**Catatan incremental:** pada refresh per jam, set `FULL_REFRESH = False` dan isi
`INCREMENTAL_WATERMARK`. Bronze memfilter berdasarkan `_ingested_at` /
`created_at` sumber, Silver dan Gold hanya membangun ulang partisi
`production_date` yang tersentuh (hari ini dan kemarin, karena shift malam
melewati tengah malam).

In [0]:
dbutils.widgets.dropdown(
    "run_mode", "full",
    ["full", "bronze_only", "silver_only", "gold_only", "suggestion_only", "maintenance"],
    "Mode eksekusi")
dbutils.widgets.text("reprocess_days", "2", "Jumlah hari yang dibangun ulang (incremental)")

RUN_MODE = dbutils.widgets.get("run_mode")
REPROCESS_DAYS = int(dbutils.widgets.get("reprocess_days"))

print(f"run_mode        = {RUN_MODE}")
print(f"reprocess_days  = {REPROCESS_DAYS} "
      f"(shift malam melewati tengah malam -> minimal 2)")
print()
print("Partisi yang akan dibangun ulang pada mode incremental:")
display(spark.sql(f"""
    SELECT explode(sequence(
        date_sub(current_date(), {REPROCESS_DAYS - 1}),
        current_date(),
        INTERVAL 1 DAY)) AS production_date
"""))

---
## 10. Ringkasan output

Semua tabel yang dihasilkan notebook ini beserta grain dan pemakaiannya.

In [0]:
summary = []
for layer in ["bronze", "silver", "gold"]:
    for t in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMAS[layer]}").collect():
        name = t["tableName"]
        fqn = f"{CATALOG}.{SCHEMAS[layer]}.{name}"
        try:
            n = spark.table(fqn).count()
            cols = len(spark.table(fqn).columns)
        except Exception:                              # noqa: BLE001
            n, cols = -1, -1
        summary.append((layer, name, n, cols))

df_summary = spark.createDataFrame(
    summary, "layer string, table_name string, row_count long, column_count int")

display(df_summary.orderBy("layer", "table_name"))

print("=" * 78)
print("UC-07 SELESAI")
print("=" * 78)
print("Gold yang dipakai dashboard:")
for t in ["gold_funnel_stage_daily", "gold_stage_reconciliation",
          "gold_production_timeseries", "gold_contractor_daily", "gold_rom_daily",
          "gold_cp_intake_daily", "gold_stockpile_balance_hourly",
          "gold_loss_anomaly", "gold_shipment_outlook", "gold_suggestion_daily",
          "gold_kpi_snapshot"]:
    print(f"  - {tbl('gold', t)}")
print()
print("Yang perlu dikonfirmasi ke tim operasional sebelum go-live:")
print("  1. OPENING_STOCK_TON  - saldo awal North Stockpile dari stock opname")
print("     (sekarang 0 -> gold_stockpile_balance_hourly bersifat saldo RELATIF).")
print("  2. SRC_URISE_TABLE / SRC_USCAVIS_TABLE - nama tabel Unity Catalog untuk Urise")
print("     dan Uscavis (sel 2.0 mencoba menemukannya otomatis; selama belum ketemu")
print("     notebook membaca file CSV di SOURCE_ROOT).")
print("  3. Stage SHIPMENT diambil dari uc.uscavis.gold_wim_cv_ratio (UC-06). Cek")
print("     kolom is_estimated di gold_funnel_stage_daily: kalau true, sumber UC-06")
print("     tidak terbaca dan angkanya masih estimasi CP intake x recovery.")
print("  4. MONTHLY_TARGET_TON / TARGET_BY_MONTH / ROM_TARGET_SHARE - ganti dengan RKAP.")
print("  5. Ambang LOSS_PCT_WARN / LOSS_PCT_CRITICAL / PAYLOAD_MIN_TON / PAYLOAD_MAX_TON.")